In [ ]:
import os, json, time, math, random, gc, re, io, sys, shutil, platform, zipfile, copy, threading, warnings, subprocess, hashlib
from types import SimpleNamespace
from collections import deque, OrderedDict, Counter

NOTEBOOK_T0 = time.time()
TIMES = {}

CFG = SimpleNamespace(
    seed=42,
    img_size=384, num_classes=5,
    class_names=['None', 'Mild', 'Moderate', 'Severe', 'Proliferative'],
    split_fracs=(0.80, 0.10, 0.10),
    eye_subset_size=10_000,
    subset_seed=42, subset_min_class_frac=0.04,
    delete_eyepacs_archives_after_cache=True,
    crop_threshold=10, bg_sigma_frac=1/30, jpeg_quality=95, pre_version='v1-foreground-crop+pad+resize+benGraham',
   
    max_cache_procs=12,
    max_fail_frac=0.02, max_missing_frac=0.01,
    aug_rotation_range_deg=(0, 360),
    aug_brightness=0.20, aug_contrast=0.20, aug_crop_scale=(0.80, 1.00), aug_crop_ratio_jitter=0.10,
    epochs=12, batch_size=64, warmup_epochs=1, min_lr_factor=0.01,
    lr_backbone=2e-4, lr_head=1e-3, weight_decay=1e-4, grad_clip=1.0,
    amp=True,
    reuse_finished=False,
    parallel_gpus=False,
    hf_push=False,
    init_micro_batch=64, max_oom_reductions=3,
    eval_batch_size=128, eval_chunk=128,
    nf_window=500, nf_frac=0.01,
    # evaluation
    ece_bins=15, referable_min_grade=2, referable_threshold=0.5,
    cv_folds=5, lr_combiner_C=1.0,
    boot_reps=1000, boot_seed=2024,
    kaggle_competitions=dict(eyepacs='diabetic-retinopathy-detection', aptos='aptos2019-blindness-detection'),
    download_retries=2,
    use_kagglehub_fallback=True,
    delete_downloads_at_end=True,
    data_root=None,
)

MODELS = OrderedDict([
    ('resnet50',        dict(display='ResNet50',        timm_id='resnet50.a1_in1k',                          ckpt='best_resnet50.pt',         channels_last=True)),
    ('efficientnet_b0', dict(display='EfficientNet-B0', timm_id='efficientnet_b0.ra_in1k',                   ckpt='best_efficientnet_b0.pt',  channels_last=True)),
    ('vit_tiny',        dict(display='ViT-Tiny',        timm_id='vit_tiny_patch16_384.augreg_in21k_ft_in1k', ckpt='best_vit_tiny.pt',         channels_last=False)),
])
MODEL_KEYS = list(MODELS.keys())

PATHS = SimpleNamespace(
    input='/kaggle/input',
    work='/kaggle/working',
)
os.makedirs(PATHS.work, exist_ok=True)
DATA_ROOT = CFG.data_root or (os.path.join('/kaggle/temp', 'dr_data') if os.path.isdir('/kaggle/temp') and os.access('/kaggle/temp', os.W_OK) else os.path.join(PATHS.work, 'dr_data'))
PATHS.data_root = DATA_ROOT
PATHS.cache_root = os.path.join(DATA_ROOT, 'cache')
PATHS.big_in_working = os.path.realpath(DATA_ROOT).startswith(os.path.realpath(PATHS.work) + os.sep)
PATHS.eyepacs_dl = os.path.join(DATA_ROOT, 'eyepacs')
PATHS.aptos_dl = os.path.join(DATA_ROOT, 'aptos')
os.makedirs(PATHS.cache_root, exist_ok=True)
PRE_SIG = hashlib.md5(json.dumps(dict(v=CFG.pre_version, size=CFG.img_size, thr=CFG.crop_threshold, sigma=CFG.bg_sigma_frac, q=CFG.jpeg_quality), sort_keys=True).encode()).hexdigest()[:8]
PATHS.cache_eyepacs = None
PATHS.cache_aptos = None

def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def make_sig(*parts):
    return hashlib.md5(json.dumps(parts, sort_keys=True, default=str).encode()).hexdigest()[:8]
PATHS.out = PATHS.work
PATHS.preds = os.path.join(PATHS.work, 'preds')
for _d in (PATHS.work, PATHS.cache_root, PATHS.preds):
    os.makedirs(_d, exist_ok=True)

print(json.dumps({k: v for k, v in vars(CFG).items() if k != 'class_names'}, indent=1, default=str))
print('cache root :', PATHS.cache_root)
print('DATA_ROOT   :', DATA_ROOT, '(datasets are fetched here only if they are not attached under', PATHS.input + ')')
print('output dir :', PATHS.out)

In [ ]:
import torch, psutil
import urllib.request, urllib.error

def fmt_bytes(n):
    for u in ('B', 'KB', 'MB', 'GB', 'TB'):
        if abs(n) < 1024 or u == 'TB':
            return f'{n:.1f} {u}'
        n /= 1024

def resource_snapshot():
    vm = psutil.virtual_memory()
    d = shutil.disk_usage(PATHS.work)
    snap = dict(cpu_ram_used_gb=round((vm.total - vm.available) / 2**30, 2), cpu_ram_total_gb=round(vm.total / 2**30, 2),
                disk_free_gb=round(d.free / 2**30, 2))
    if torch.cuda.is_available():
        snap['gpu_mem_alloc_gb'] = [round(torch.cuda.memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())]
    return snap

def usable_cpus():
    n = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 4)
    try:
        quota, period = open('/sys/fs/cgroup/cpu.max').read().split()
        if quota != 'max': n = min(n, max(1, int(int(quota) / int(period))))
    except Exception:
        pass
    return max(1, n)
USABLE_CPUS = usable_cpus()

NUM_GPUS = torch.cuda.device_count()
print('Python      :', sys.version.split()[0], '|', platform.platform())
print('PyTorch     :', torch.__version__)
print('CUDA        :', torch.version.cuda, '| cuDNN', torch.backends.cudnn.version())
print('GPUs        :', NUM_GPUS)
for _i in range(NUM_GPUS):
    _p = torch.cuda.get_device_properties(_i)
    print(f'  GPU {_i}: {_p.name}  {_p.total_memory / 2**30:.1f} GB')
print('CPU cores   :', os.cpu_count(), '(host-reported) |', USABLE_CPUS, 'usable | RAM total', fmt_bytes(psutil.virtual_memory().total))
PARALLEL = bool(CFG.parallel_gpus and NUM_GPUS >= 2)
if NUM_GPUS == 0:
    raise RuntimeError('No CUDA GPU visible. Enable a GPU accelerator (Settings -> Accelerator -> a T4 or any other CUDA GPU; one GPU is enough) and re-run.')
TRAIN_PLAN = 'parallel (GPU0: ResNet50 -> ViT-Tiny, GPU1: EfficientNet-B0)' if PARALLEL else 'sequential: ResNet50 -> EfficientNet-B0 -> ViT-Tiny on cuda:0'
print('Training plan:', TRAIN_PLAN)

def internet_status():
    out = {}
    for name, url in (('huggingface.co  (pretrained timm weights)', 'https://huggingface.co'), ('kaggle.com      (dataset download via the Kaggle API)', 'https://www.kaggle.com')):
        try:
            urllib.request.urlopen(urllib.request.Request(url, method='HEAD'), timeout=8); out[name] = True
        except urllib.error.HTTPError:
            out[name] = True
        except Exception:
            out[name] = False
    return out

print('\nINTERNET IS REQUIRED: the pretrained weights (timm, pretrained=True) are downloaded at run time, and the datasets are downloaded through the Kaggle API if they are not attached.')
_net = internet_status()
for _k, _v in _net.items():
    print(f'  {_k}: {"reachable" if _v else "NOT reachable"}')
if not any(v for k, v in _net.items() if k.startswith('huggingface')):
    raise RuntimeError('Internet is not reachable, but the pretrained model weights must be downloaded. Enable Internet in the notebook settings (Settings -> Internet -> On) and re-run.')
print('Dataset sources are resolved in Cell 5: attached /kaggle/input -> existing verified download -> Kaggle API download.')

In [ ]:
# ===== CELL 4 — PACKAGE IMPORTS + REPRODUCIBILITY =====
def _ensure(pip_name, module_name=None):
    import importlib
    try:
        importlib.import_module(module_name or pip_name)
    except ImportError:
        print('installing', pip_name)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

_ensure('timm')
_ensure('opencv-python-headless', 'cv2')

import numpy as np
import pandas as pd
import cv2
import timm
import matplotlib
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, roc_auc_score, precision_recall_fscore_support, confusion_matrix

warnings.filterwarnings('ignore', message='.*lr_scheduler.*')
warnings.filterwarnings('ignore', category=FutureWarning)
cv2.setNumThreads(1)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

def seed_worker(worker_id):
    s = (torch.initial_seed() + worker_id) % 2**32
    np.random.seed(s); random.seed(s); cv2.setNumThreads(0)

set_seed(CFG.seed)
torch.backends.cudnn.benchmark = True      # fixed input shape -> faster. Seed-controlled, NOT bit-wise deterministic (cuDNN autotuning / fp16 / atomics)

print('random seed   :', CFG.seed)
print('PyTorch       :', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU(s)        :', [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)], '| count', NUM_GPUS)
print('timm          :', timm.__version__, '| OpenCV', cv2.__version__)
if not timm.__version__.startswith(('0.9', '1.0')):
    print('NOTE: developed against timm 0.9/1.0; the structural assertions in Cells 18-22 will stop the run if layer names/depths differ in this version.')
print('Pretrained model identifiers:')
for k, v in MODELS.items():
    print(f'  {v["display"]:16s} -> {v["timm_id"]}')

In [ ]:
import urllib.request, urllib.error, glob
import importlib, importlib.metadata
IMG_EXT = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')

class AcquisitionError(RuntimeError):
    """Raised with an actionable message when a dataset can neither be found nor downloaded."""

def ensure_pkg(pip_name, module_name):
    import importlib.util
    if module_name in sys.modules or importlib.util.find_spec(module_name) is not None:
        return True
    print('installing', pip_name)
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name]); return True
    except Exception as e:
        print(f'  could not install {pip_name}: {type(e).__name__}'); return False

def head_tail_md5(path, n=1 << 20):
    h = hashlib.md5(); size = os.path.getsize(path)
    with open(path, 'rb') as f:
        h.update(f.read(n)); f.seek(max(0, size - n)); h.update(f.read(n))
    return h.hexdigest()

class MultiPartFile(io.RawIOBase):
    def __init__(self, paths):
        super().__init__()
        self.paths = list(paths)
        self.sizes = [os.path.getsize(p) for p in self.paths]
        self.offs = np.concatenate([[0], np.cumsum(self.sizes)]).astype(np.int64)
        self.total = int(self.offs[-1]); self.pos = 0; self._fh = None; self._fi = -1
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else (self.pos + off if whence == 1 else self.total + off)
        self.pos = max(0, min(self.pos, self.total))
        return self.pos
    def readinto(self, b):
        mv = memoryview(b).cast('B'); want = len(mv); n = 0
        while n < want and self.pos < self.total:
            i = int(np.searchsorted(self.offs, self.pos, side='right') - 1)
            if i != self._fi:
                if self._fh is not None: self._fh.close()
                self._fh = open(self.paths[i], 'rb'); self._fi = i
            within = self.pos - int(self.offs[i])
            take = min(want - n, self.sizes[i] - within)
            self._fh.seek(within)
            got = self._fh.readinto(mv[n:n + take])
            if not got: break
            n += got; self.pos += got
        return n
    def close(self):
        if self._fh is not None: self._fh.close(); self._fh = None
        super().close()

def open_zip(paths):
    if len(paths) == 1:
        return zipfile.ZipFile(paths[0])
    return zipfile.ZipFile(io.BufferedReader(MultiPartFile(paths), buffer_size=4 << 20))

def read_csv_any(path):
    if path.lower().endswith('.zip'):
        with zipfile.ZipFile(path) as z:
            name = next(n for n in z.namelist() if n.lower().endswith('.csv'))
            with z.open(name) as f:
                return pd.read_csv(f)
    return pd.read_csv(path)

KAGGLE_AUTH_HELP = ('Kaggle API authentication is required to download the data automatically. Provide credentials with ONE of: '
                    '(a) Notebook -> Add-ons -> Secrets: add secrets named KAGGLE_USERNAME and KAGGLE_KEY (create a token at kaggle.com -> Settings -> API) '
                    'and attach them to this notebook; (b) a ~/.kaggle/kaggle.json file; or (c) attach the competition manually via Add Input -> Competitions.')

def explain_kaggle_error(label, comp, e):
    msg, low = f'{type(e).__name__}: {str(e)[:300]}', str(e).lower()
    if '403' in low or 'forbidden' in low:
        hint = (f'Access to the competition "{comp}" was refused (HTTP 403). With the same Kaggle account as the API credentials, open '
                f'https://www.kaggle.com/c/{comp}, click "Join Competition" and accept the rules (Kaggle requires this and it cannot be automated), then re-run.')
    elif any(w in low for w in ('401', 'unauthorized', 'could not find kaggle.json', 'credentials', 'authenticate')):
        hint = KAGGLE_AUTH_HELP
    elif '404' in low or 'not found' in low:
        hint = f'The competition or file was not found on Kaggle (check the competition id "{comp}").'
    elif any(w in low for w in ('connection', 'timed out', 'timeout', 'name or service', 'resolve', 'network', 'unreachable')):
        hint = 'Internet access appears to be unavailable. Enable Internet in the notebook settings (it is also needed for the pretrained model weights).'
    else:
        hint = 'See the underlying error below.'
    return f'{label} is not attached and could not be downloaded automatically. Kaggle API authentication or competition access is required. {hint} [underlying error: {msg}]'

def load_kaggle_secrets_into_env():
    loaded = []
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
    except Exception:
        return loaded
    for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY', 'KAGGLE_API_TOKEN'):
        if os.environ.get(k):
            continue
        try:
            v = us.get_secret(k)
        except Exception:
            continue
        if v:
            os.environ[k] = v; loaded.append(k)
    return loaded

def get_kaggle_api():
    loaded = load_kaggle_secrets_into_env()
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi      # note: old kaggle versions authenticate at import time
        api = KaggleApi(); api.authenticate()
    except ImportError as e:
        raise AcquisitionError(f'The "kaggle" python package is not importable ({e}). Run: pip install kaggle') from e
    except (Exception, SystemExit) as e:
        raise AcquisitionError(f'Kaggle API authentication failed ({type(e).__name__}: {str(e)[:200]}). ' + KAGGLE_AUTH_HELP) from e
    print('Kaggle API authenticated' + (f' (using Kaggle Secrets: {", ".join(loaded)})' if loaded else ''))
    return api

def _to_bytes(v):
    if v is None: return None
    if isinstance(v, (int, float)): return int(v)
    m = re.match(r'^\s*([\d.]+)\s*([KMGT]?B)\s*$', str(v), re.I)
    if m: return int(float(m.group(1)) * 1024 ** ['B', 'KB', 'MB', 'GB', 'TB'].index(m.group(2).upper()))
    return int(v) if str(v).strip().isdigit() else None

def list_competition_files(api, comp, max_pages=3):
    files, token = [], None
    for _ in range(max_pages):
        try:
            res = api.competition_list_files(comp, page_token=token, page_size=200)
        except TypeError:
            res = api.competition_list_files(comp)
        items = getattr(res, 'files', res)
        for f in items:
            size = next((_to_bytes(getattr(f, a, None)) for a in ('totalBytes', 'total_bytes', 'size') if getattr(f, a, None) is not None), None)
            files.append((getattr(f, 'name', str(f)), size))
        token = getattr(res, 'next_page_token', None) or getattr(res, 'nextPageToken', None)
        if not token: break
    return files

def preflight_competition(api, label, comp):
    try:
        files = list_competition_files(api, comp, max_pages=1)
    except (Exception, SystemExit) as e:
        raise AcquisitionError(explain_kaggle_error(label, comp, e)) from e
    if not files:
        raise AcquisitionError(explain_kaggle_error(label, comp, RuntimeError('403 the competition returned an empty file list (rules not accepted?)')))
    return files

def write_manifest(d, comp, files, **extra):
    with open(os.path.join(d, 'download_manifest.json'), 'w') as f:
        json.dump(dict(competition=comp, complete=True, files=files, **extra), f, indent=1)

def read_manifest(d):
    try:
        with open(os.path.join(d, 'download_manifest.json')) as f:
            m = json.load(f)
        return m if m.get('complete') else None
    except Exception:
        return None

EYE_FILE_RE = re.compile(r'^(trainLabels\.csv(\.zip)?|train\.zip(\.\d+)?)$', re.I)
EYE_FALLBACK_FILES = [('trainLabels.csv.zip', None)] + [(f'train.zip.{i:03d}', None) for i in range(1, 6)]

def eyepacs_download_valid():
    m = read_manifest(PATHS.eyepacs_dl)
    if not m: return False
    try:
        ok = all(os.path.getsize(os.path.join(PATHS.eyepacs_dl, n)) == s for n, s in m['files'].items()) and os.path.exists(os.path.join(PATHS.eyepacs_dl, 'trainLabels.csv'))
        return ok and all(head_tail_md5(os.path.join(PATHS.eyepacs_dl, n)) == h for n, h in m.get('hashes', {}).items())
    except OSError:
        return False

def aptos_download_valid():
    m = read_manifest(PATHS.aptos_dl)
    if not m: return False
    try:
        n = sum(1 for f in os.listdir(os.path.join(PATHS.aptos_dl, 'train_images')) if f.lower().endswith(IMG_EXT))
        return os.path.exists(os.path.join(PATHS.aptos_dl, 'train.csv')) and n == m.get('n_images')
    except OSError:
        return False

def _is_fatal(e):
    low = str(e).lower()
    return any(w in low for w in ('401', '403', '404', 'forbidden', 'unauthorized', 'not found', 'credentials'))

def api_download_file(api, comp, name, dest_dir, expected_size):
    os.makedirs(dest_dir, exist_ok=True)
    target = os.path.join(dest_dir, name)
    if expected_size is not None and os.path.exists(target) and os.path.getsize(target) == expected_size:
        print(f'  {name}: already downloaded ({fmt_bytes(expected_size)})'); return target
    last = None
    for attempt in range(1, CFG.download_retries + 2):
        for stale in (target, target + '.zip'):
            if os.path.exists(stale): os.remove(stale)
        try:
            t0 = time.time()
            print(f'  downloading {name}' + (f' ({fmt_bytes(expected_size)})' if expected_size else '') + ' ...', flush=True)
            api.competition_download_file(comp, name, path=dest_dir, force=True, quiet=True)
            got = target if os.path.exists(target) else (target + '.zip' if os.path.exists(target + '.zip') else None)
            if got is None: raise AcquisitionError(f'{name}: the API reported success but no file appeared in {dest_dir}')
            if got != target: os.replace(got, target)
            size = os.path.getsize(target)
            if size == 0 or (expected_size is not None and size != expected_size):
                raise AcquisitionError(f'{name}: downloaded {size} bytes but expected {expected_size} (partial download)')
            print(f'  {name}: OK {fmt_bytes(size)} in {(time.time() - t0) / 60:.1f} min', flush=True)
            return target
        except (Exception, SystemExit) as e:
            last = e
            if _is_fatal(e) or attempt > CFG.download_retries: break
            print(f'  {name}: attempt {attempt} failed ({type(e).__name__}: {str(e)[:120]}); retrying', flush=True); time.sleep(5 * attempt)
    raise last if isinstance(last, AcquisitionError) else AcquisitionError(f'{name}: download failed ({type(last).__name__}: {str(last)[:200]})')

def verify_eyepacs_download(d, sizes):
    for n, s in sizes.items():
        p = os.path.join(d, n)
        if not os.path.isfile(p): raise AcquisitionError(f'EyePACS: expected file {n} is missing')
        got = os.path.getsize(p)
        if got == 0 or (s is not None and got != s): raise AcquisitionError(f'EyePACS: {n} has {got} bytes, expected {s} (partial download)')
    lc, lz = os.path.join(d, 'trainLabels.csv'), os.path.join(d, 'trainLabels.csv.zip')
    if not os.path.exists(lc):
        if not os.path.exists(lz): raise AcquisitionError('EyePACS: trainLabels.csv(.zip) is missing')
        if not zipfile.is_zipfile(lz): raise AcquisitionError('EyePACS: trainLabels.csv.zip is not a valid zip (partial download?)')
        with zipfile.ZipFile(lz) as z:
            member = next(m for m in z.namelist() if m.lower().endswith('.csv'))
            with z.open(member) as src, open(lc, 'wb') as dst: shutil.copyfileobj(src, dst)
    lab = pd.read_csv(lc)
    if not {'image', 'level'} <= set(lab.columns) or len(lab) < 1000:
        raise AcquisitionError(f'EyePACS: trainLabels.csv unreadable or unexpected (columns {list(lab.columns)}, rows {len(lab)})')
    parts = sorted((n for n in sizes if re.match(r'^train\.zip\.\d+$', n, re.I)), key=lambda s: int(s.rsplit('.', 1)[1]))
    single = [n for n in sizes if n.lower() == 'train.zip']
    if not parts and not single: raise AcquisitionError('EyePACS: no train archive (train.zip.00N / train.zip) among the downloaded files')
    if parts:
        nums = [int(p.rsplit('.', 1)[1]) for p in parts]
        if nums != list(range(nums[0], nums[0] + len(nums))): raise AcquisitionError(f'EyePACS: train archive parts are not contiguous: {nums}')
        with open(os.path.join(d, parts[0]), 'rb') as f:
            if f.read(4) != b'PK\x03\x04': raise AcquisitionError('EyePACS: the first archive part does not start with a zip header')
        last = os.path.join(d, parts[-1])
        with open(last, 'rb') as f:
            f.seek(max(0, os.path.getsize(last) - 131072)); tail = f.read()
        if b'PK\x05\x06' not in tail and b'PK\x06\x06' not in tail:
            raise AcquisitionError('EyePACS: the last archive part has no zip end-of-archive record (truncated download)')
    elif not zipfile.is_zipfile(os.path.join(d, single[0])):
        raise AcquisitionError('EyePACS: train.zip is not a valid zip')
    paths = [os.path.join(d, n) for n in (parts or single)]
    note = ''
    try:
        with open_zip(paths) as z:
            infos = [i for i in z.infolist() if not i.is_dir() and i.filename.lower().endswith(IMG_EXT)]
            if len(infos) < 1000: raise AcquisitionError(f'EyePACS: the train archive lists only {len(infos)} images')
            for i in infos[::max(1, len(infos) // 3)][:3]:
                if cv2.imdecode(np.frombuffer(z.read(i), np.uint8), cv2.IMREAD_COLOR) is None:
                    raise AcquisitionError(f'EyePACS: sample image {i.filename} could not be decoded')
        note = f'{len(infos)} images listed in the archive, 3 sample images decoded'
    except zipfile.BadZipFile as e:
        note = f'WARNING: the concatenated archive could not be opened by python ({e}); Cell 6 will use its 7-Zip fallback'
    print(f'EyePACS download verified: {len(lab)} labelled rows, {len(parts) or 1} archive part(s), {note}')

def acquire_eyepacs_api(api, listing):
    comp, d = CFG.kaggle_competitions['eyepacs'], PATHS.eyepacs_dl
    os.makedirs(d, exist_ok=True)
    wanted = [(n, s) for n, s in (listing or []) if EYE_FILE_RE.match(n)] or EYE_FALLBACK_FILES
    for n, s in wanted: api_download_file(api, comp, n, d, s)
    sizes = {n: (s if s is not None else os.path.getsize(os.path.join(d, n))) for n, s in wanted}
    verify_eyepacs_download(d, sizes)
    sizes_all = dict(sizes); sizes_all['trainLabels.csv'] = os.path.getsize(os.path.join(d, 'trainLabels.csv'))
    write_manifest(d, comp, sizes_all, hashes={n: head_tail_md5(os.path.join(d, n)) for n in sizes_all})

def acquire_eyepacs_hub():
    comp, d = CFG.kaggle_competitions['eyepacs'], PATHS.eyepacs_dl
    os.makedirs(d, exist_ok=True)
    os.environ['KAGGLEHUB_CACHE'] = os.path.join(PATHS.data_root, '_kagglehub_cache')
    import kagglehub
    for n, _ in EYE_FALLBACK_FILES:
        p = kagglehub.competition_download(comp, path=n)
        if os.path.isdir(p): p = os.path.join(p, n)
        dst = os.path.join(d, n)
        if os.path.abspath(p) != os.path.abspath(dst): shutil.move(p, dst)
    sizes = {n: os.path.getsize(os.path.join(d, n)) for n, _ in EYE_FALLBACK_FILES}
    verify_eyepacs_download(d, sizes)
    sizes['trainLabels.csv'] = os.path.getsize(os.path.join(d, 'trainLabels.csv'))
    write_manifest(d, comp, sizes, hashes={n: head_tail_md5(os.path.join(d, n)) for n in sizes})
    shutil.rmtree(os.environ['KAGGLEHUB_CACHE'], ignore_errors=True)

def extract_aptos_zip(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        wanted = []
        for info in z.infolist():
            if info.is_dir(): continue
            parts = info.filename.replace('\\', '/').split('/')
            if parts[-1].lower() == 'train.csv' and 'train_images' not in parts[:-1]: wanted.append((info, 'train.csv'))
            elif 'train_images' in parts[:-1] and parts[-1].lower().endswith(IMG_EXT): wanted.append((info, 'train_images/' + parts[-1]))
        n_img = sum(1 for _, rel in wanted if rel.startswith('train_images/'))
        if not any(rel == 'train.csv' for _, rel in wanted) or n_img < 1000:
            raise AcquisitionError(f'APTOS: the downloaded zip does not contain train.csv + train_images (found {n_img} train images)')
        for info, rel in wanted:
            out = os.path.join(dest, rel); os.makedirs(os.path.dirname(out), exist_ok=True)
            with z.open(info) as src, open(out, 'wb') as dst: shutil.copyfileobj(src, dst, 1 << 20)
            if os.path.getsize(out) != info.file_size: raise AcquisitionError(f'APTOS: extracted {rel} has the wrong size')
    return n_img

def verify_aptos_dir(d):
    ap = pd.read_csv(os.path.join(d, 'train.csv'))
    if not {'id_code', 'diagnosis'} <= set(ap.columns): raise AcquisitionError(f'APTOS: train.csv has columns {list(ap.columns)}, expected id_code and diagnosis')
    imgs = {os.path.splitext(f)[0] for f in os.listdir(os.path.join(d, 'train_images')) if f.lower().endswith(IMG_EXT)}
    found = sum(1 for i in ap.id_code.astype(str) if i in imgs)
    if found < 0.99 * len(ap): raise AcquisitionError(f'APTOS: only {found}/{len(ap)} labelled images were extracted')
    for i in list(ap.id_code.astype(str))[:3]:
        f = next(x for x in os.listdir(os.path.join(d, 'train_images')) if os.path.splitext(x)[0] == i)
        if cv2.imread(os.path.join(d, 'train_images', f)) is None: raise AcquisitionError(f'APTOS: sample image {f} could not be decoded')
    print(f'APTOS download verified: {len(ap)} labelled rows, {found} images present, 3 sample images decoded')
    return len(imgs)

def _finish_aptos(n_images, comp):
    write_manifest(PATHS.aptos_dl, comp, {'train.csv': os.path.getsize(os.path.join(PATHS.aptos_dl, 'train.csv'))}, n_images=n_images)

def acquire_aptos_api(api):
    comp, d = CFG.kaggle_competitions['aptos'], PATHS.aptos_dl
    zdir = os.path.join(d, '_download'); shutil.rmtree(zdir, ignore_errors=True); os.makedirs(zdir, exist_ok=True)
    print('  downloading the APTOS competition archive ...', flush=True); t0 = time.time()
    try:
        api.competition_download_files(comp, path=zdir, force=True, quiet=True)
    except (Exception, SystemExit) as e:
        raise AcquisitionError(explain_kaggle_error('APTOS 2019', comp, e)) from e
    zips = sorted(glob.glob(os.path.join(zdir, '*.zip')), key=os.path.getsize, reverse=True)
    if not zips: raise AcquisitionError(f'APTOS: no zip appeared in {zdir} after the API download')
    print(f'  APTOS archive downloaded: {fmt_bytes(os.path.getsize(zips[0]))} in {(time.time() - t0) / 60:.1f} min; extracting train.csv + train_images ...', flush=True)
    try:
        extract_aptos_zip(zips[0], d)
    except zipfile.BadZipFile as e:
        raise AcquisitionError(f'APTOS: the downloaded archive is not a valid zip (partial download?): {e}') from e
    shutil.rmtree(zdir, ignore_errors=True)
    _finish_aptos(verify_aptos_dir(d), comp)

def acquire_aptos_hub():
    comp, d = CFG.kaggle_competitions['aptos'], PATHS.aptos_dl
    os.makedirs(d, exist_ok=True)
    os.environ['KAGGLEHUB_CACHE'] = os.path.join(PATHS.data_root, '_kagglehub_cache')
    import kagglehub
    p = kagglehub.competition_download(comp)
    zips = [p] if (os.path.isfile(p) and p.lower().endswith('.zip')) else sorted(glob.glob(os.path.join(p, '**', '*.zip'), recursive=True), key=os.path.getsize, reverse=True)
    if zips:
        extract_aptos_zip(zips[0], d)
    else:
        csvs = [os.path.join(dp, 'train.csv') for dp, dn, fn in os.walk(p) if 'train.csv' in fn and 'train_images' not in dp]
        imgdirs = [os.path.join(dp, 'train_images') for dp, dn, fn in os.walk(p) if 'train_images' in dn]
        if not csvs or not imgdirs: raise AcquisitionError('APTOS: kagglehub finished but train.csv / train_images were not found')
        shutil.move(csvs[0], os.path.join(d, 'train.csv')); shutil.move(imgdirs[0], os.path.join(d, 'train_images'))
    shutil.rmtree(os.environ['KAGGLEHUB_CACHE'], ignore_errors=True)
    _finish_aptos(verify_aptos_dir(d), comp)

_KAGGLE_CLIENT_CHECKED = False
def ensure_kaggle_client():
    global _KAGGLE_CLIENT_CHECKED
    if _KAGGLE_CLIENT_CHECKED: return
    _KAGGLE_CLIENT_CHECKED = True
    def ver():
        try: return importlib.metadata.version('kaggle')
        except importlib.metadata.PackageNotFoundError: return None
    def vt(v): return tuple(int(x) for x in re.findall(r'[0-9]+', v)[:3])
    installed, latest = ver(), None
    try:
        with urllib.request.urlopen('https://pypi.org/pypi/kaggle/json', timeout=10) as r: latest = json.load(r)['info']['version']
    except Exception:
        pass
    if installed is None or (latest is not None and vt(installed) < vt(latest)):
        if 'kaggle' in sys.modules:
            print(f'kaggle client {installed} is already imported in this kernel; restart the kernel to update it to {latest}')
        else:
            try:
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kaggle']); importlib.invalidate_caches()
            except Exception as e:
                print(f'  could not update the kaggle client ({type(e).__name__})')
    print(f'kaggle client version: {ver()} (latest on PyPI: {latest})')

ACQ_METHOD = {}

def _with_fallback(label, comp, api_fn, hub_fn, api_err, key):
    errs = []
    if api_fn is not None:
        try:
            api_fn(); ACQ_METHOD[key] = 'Kaggle API'; return
        except (AcquisitionError, Exception, SystemExit) as e:
            errs.append(('Kaggle API', e))
    elif api_err is not None:
        errs.append(('Kaggle API', api_err))
    if CFG.use_kagglehub_fallback:
        try:
            hub_fn(); ACQ_METHOD[key] = 'kagglehub (Kaggle API)'; return
        except (Exception, SystemExit) as e:
            errs.append(('kagglehub', e))
    first = errs[0][1]
    head = str(first) if isinstance(first, AcquisitionError) and 'not attached' in str(first) else explain_kaggle_error(label, comp, first)
    raise AcquisitionError(head + ' | other strategies tried: ' + '; '.join(f'{n}: {type(e).__name__}: {str(e)[:150]}' for n, e in errs[1:]))

def acquire_missing(need_eye, need_aptos):
    comps = CFG.kaggle_competitions
    ensure_kaggle_client()
    if CFG.use_kagglehub_fallback: ensure_pkg('kagglehub', 'kagglehub')
    api, api_err, eye_listing = None, None, None
    try:
        api = get_kaggle_api()
    except AcquisitionError as e:
        api_err = e
        print('Kaggle API not authenticated:', str(e)[:200] + ('\n -> will try kagglehub (automatic authentication inside Kaggle notebooks) as a best-effort fallback' if CFG.use_kagglehub_fallback else ''))
    if api is not None:
        if need_aptos: preflight_competition(api, 'APTOS 2019', comps['aptos'])
        if need_eye: eye_listing = preflight_competition(api, 'EyePACS', comps['eyepacs'])
        print('Kaggle competition access verified for:', ', '.join(n for n, f in (('APTOS 2019', need_aptos), ('EyePACS', need_eye)) if f))
    if need_aptos:
        print('\nAcquiring APTOS 2019 ->', PATHS.aptos_dl)
        _with_fallback('APTOS 2019', comps['aptos'], (lambda: acquire_aptos_api(api)) if api else None, acquire_aptos_hub, api_err, 'aptos')
    if need_eye:
        print('\nAcquiring EyePACS (trainLabels + training archives only; the unlabelled test archives are never downloaded) ->', PATHS.eyepacs_dl)
        _with_fallback('EyePACS', comps['eyepacs'], (lambda: acquire_eyepacs_api(api, eye_listing)) if api else None, acquire_eyepacs_hub, api_err, 'eyepacs')

In [ ]:
def scan_tree(root):
    non_img, img_index, img_dirs = [], {}, Counter()
    for dp, dn, fn in os.walk(root, followlinks=True):
        for f in fn:
            p = os.path.join(dp, f)
            if f.lower().endswith(IMG_EXT):
                img_dirs[dp] += 1
                img_index.setdefault(os.path.splitext(f)[0], []).append(p)
            else:
                try:
                    non_img.append((p, os.path.getsize(p)))
                except OSError:
                    pass
    return non_img, img_index, img_dirs

NON_IMG, IMG_INDEX, IMG_DIRS = [], {}, Counter()
def merge_scan(t):
    NON_IMG.extend(t[0]); IMG_DIRS.update(t[2])
    for stem, ps in t[1].items():
        IMG_INDEX.setdefault(stem, []).extend(ps)

def _base(p): return os.path.basename(p).lower()
_zip_re = re.compile(r'\.zip(\.\d+)?$', re.I)
def _is_train_archive(p):
    b = _base(p)
    return bool(_zip_re.search(b)) and 'train' in b and not any(w in b for w in ('test', 'label', 'sample'))

def discover_eyepacs(non_img, img_dirs):
    lab = sorted((p for p, s in non_img if _base(p) in ('trainlabels.csv', 'trainlabels.csv.zip')), key=lambda p: (not _base(p).endswith('.csv'), p))
    if not lab: return None
    labels, ep_dir = lab[0], os.path.dirname(lab[0])
    arch = [p for p, s in non_img if _is_train_archive(p)]
    same = [p for p in arch if os.path.dirname(p) == ep_dir]
    arch = same if same else [p for p in arch if 'aptos' not in p.lower()]
    volumes, groups = [], {}
    for p in arch:
        m = re.match(r'^(?P<base>.+\.zip)\.(?P<n>\d+)$', p, re.I)
        if m: groups.setdefault(m.group('base'), []).append((int(m.group('n')), p))
        else: volumes.append(dict(kind='zip', paths=[p]))
    for base, parts in groups.items():
        parts.sort(); nums = [n for n, _ in parts]
        if nums != list(range(nums[0], nums[0] + len(nums))):
            print(f'  (multipart archive {base} has non-contiguous parts {nums} - treating this EyePACS copy as incomplete)'); return None
        volumes.append(dict(kind='zip', paths=[p for _, p in parts]))
    if not volumes:
        for d, n in img_dirs.items():
            if n >= 1000 and 'train' in d.lower() and 'test' not in d.lower() and 'aptos' not in d.lower():
                volumes.append(dict(kind='dir', root=d))
    return dict(labels=labels, dir=ep_dir, volumes=volumes) if volumes else None

def discover_aptos(non_img):
    best, best_key = None, (False, -1)
    for p, s in non_img:
        if p.lower().endswith('.csv') and s < 5_000_000:
            try: head = pd.read_csv(p, nrows=3)
            except Exception: continue
            if {'id_code', 'diagnosis'} <= set(head.columns):
                n = sum(1 for _ in open(p, 'rb')) - 1
                if ('aptos' in p.lower(), n) > best_key: best, best_key = (p, n), ('aptos' in p.lower(), n)
    return best

def aptos_has_images(csv_path, img_dirs):
    root = os.path.dirname(csv_path)
    return sum(n for d, n in img_dirs.items() if d.startswith(root)) >= 100

def find_competition_dir(slug):
    for c in (os.path.join(PATHS.input, 'competitions', slug), os.path.join(PATHS.input, slug)):
        if os.path.isdir(c): return c
    if os.path.isdir(PATHS.input):
        base_depth = PATHS.input.rstrip(os.sep).count(os.sep)
        for dp, dn, fn in os.walk(PATHS.input, followlinks=True):
            if os.path.basename(dp) == slug: return dp
            if dp.count(os.sep) - base_depth >= 3: dn[:] = []
    return None

def detect_attached_aptos():
    d = find_competition_dir(CFG.kaggle_competitions['aptos'])
    if d is None: return None
    flags = {n: os.path.exists(os.path.join(d, n)) for n in ('train.csv', 'train_images', 'test.csv', 'test_images')}
    print(f'APTOS 2019 detected: {d}')
    for n, ok in flags.items(): print(f'    {n:14s}: {"FOUND" if ok else "NOT FOUND"}')
    if not (flags['train.csv'] and flags['train_images']):
        print('    train.csv and train_images/ are both required -> this copy cannot be used'); return None
    csv = os.path.join(d, 'train.csv')
    if not {'id_code', 'diagnosis'} <= set(pd.read_csv(csv, nrows=3).columns):
        print('    train.csv has no id_code / diagnosis columns -> this copy cannot be used'); return None
    return csv, sum(1 for _ in open(csv, 'rb')) - 1

def detect_attached_eyepacs():
    d = find_competition_dir(CFG.kaggle_competitions['eyepacs'])
    if d is None: return None
    print(f'EyePACS (Diabetic Retinopathy Detection) detected: {d}')
    t = scan_tree(d); merge_scan(t)
    ep = discover_eyepacs(t[0], t[2])
    if ep is None:
        lab = sorted(os.path.basename(q) for q, s in t[0] if _base(q) in ('trainlabels.csv', 'trainlabels.csv.zip'))
        arch = sorted(os.path.basename(q) for q, s in t[0] if _is_train_archive(q))
        raise AcquisitionError(f'EyePACS is attached at {d} (so it is NOT downloaded) but it cannot be used as it is. labels file: {lab or "NOT FOUND (expected trainLabels.csv.zip)"}; '
                               f'training archives: {arch or "NOT FOUND (expected train.zip.001, train.zip.002, ...)"}. The archive parts must be complete and consecutive.')
    return ep

print('Kaggle Input root:', PATHS.input, '->', 'present' if os.path.isdir(PATHS.input) else 'NOT present')
if os.path.isdir(PATHS.input):
    print('  top level   :', sorted(os.listdir(PATHS.input)) or '(empty)')
    if os.path.isdir(os.path.join(PATHS.input, 'competitions')): print('  competitions:', sorted(os.listdir(os.path.join(PATHS.input, 'competitions'))))
AP = detect_attached_aptos()
EP = detect_attached_eyepacs()
if (AP is None or EP is None) and os.path.isdir(PATHS.input):
    INPUT_SCAN = scan_tree(PATHS.input); merge_scan(INPUT_SCAN)
    EP = EP or discover_eyepacs(INPUT_SCAN[0], INPUT_SCAN[2])
    if AP is None:
        AP = discover_aptos(INPUT_SCAN[0])
        if AP is not None and not aptos_has_images(AP[0], INPUT_SCAN[2]):
            print('  APTOS csv found in Kaggle Input but its images are missing -> will not use it'); AP = None
EYEPACS_MODE = 'ATTACHED KAGGLE INPUT' if EP else None
APTOS_MODE = 'ATTACHED KAGGLE INPUT' if AP else None
EYEPACS_SOURCE = f'{EP["dir"]}  (ATTACHED KAGGLE INPUT)' if EP else None
APTOS_SOURCE = f'{os.path.dirname(AP[0])}  (ATTACHED KAGGLE INPUT)' if AP else None
print()
print('ACQUISITION STATUS (before any download)')
print(f'  APTOS attached   : {"YES -> " + os.path.dirname(AP[0]) if AP else "NO"}')
print(f'  EyePACS attached : {"YES -> " + EP["dir"] if EP else "NO"}')
print(f'  APTOS acquisition   : {"ATTACHED INPUT (download skipped, no Kaggle API / kagglehub call)" if AP else "existing verified download, otherwise Kaggle API download"}')
print(f'  EyePACS acquisition : {"ATTACHED INPUT (download skipped, no Kaggle API / kagglehub call)" if EP else "existing verified download, otherwise Kaggle API download"}')

if EP is None and eyepacs_download_valid():
    t = scan_tree(PATHS.eyepacs_dl); merge_scan(t); EP = discover_eyepacs(t[0], t[2])
    if EP: EYEPACS_SOURCE, EYEPACS_MODE = f'{PATHS.eyepacs_dl}  (existing verified download - not downloaded again)', 'EXISTING VERIFIED DOWNLOAD'
if AP is None and aptos_download_valid():
    t = scan_tree(PATHS.aptos_dl); merge_scan(t); AP = discover_aptos(t[0])
    if AP: APTOS_SOURCE, APTOS_MODE = f'{PATHS.aptos_dl}  (existing verified download - not downloaded again)', 'EXISTING VERIFIED DOWNLOAD'

if EP is None or AP is None:
    print('\nMissing -> ' + ', '.join(n for n, m in (('EyePACS', EP is None), ('APTOS 2019', AP is None)) if m) +
          '. Not attached and no valid earlier download, so they will be downloaded through the Kaggle API (Internet required).')
    _t_dl = time.time()
    acquire_missing(need_eye=EP is None, need_aptos=AP is None)
    TIMES['download_s'] = time.time() - _t_dl
    if EP is None:
        t = scan_tree(PATHS.eyepacs_dl); merge_scan(t); EP = discover_eyepacs(t[0], t[2])
        if EP is None: raise AcquisitionError(f'EyePACS download finished but trainLabels.csv + the training archives were not found in {PATHS.eyepacs_dl}.')
        EYEPACS_SOURCE, EYEPACS_MODE = f'downloaded through {ACQ_METHOD.get("eyepacs", "Kaggle API")} -> {PATHS.eyepacs_dl}', 'KAGGLE API DOWNLOAD'
    if AP is None:
        t = scan_tree(PATHS.aptos_dl); merge_scan(t); AP = discover_aptos(t[0])
        if AP is None or not aptos_has_images(AP[0], t[2]): raise AcquisitionError(f'APTOS download finished but train.csv + train_images were not found in {PATHS.aptos_dl}.')
        APTOS_SOURCE, APTOS_MODE = f'downloaded through {ACQ_METHOD.get("aptos", "Kaggle API")} -> {PATHS.aptos_dl}', 'KAGGLE API DOWNLOAD'

TIMES.setdefault('download_s', 0.0)
LABELS_PATH, EYEPACS_DIR, VOLUMES = EP['labels'], EP['dir'], EP['volumes']
APTOS_CSV, _best_n = AP
print()
print('ACQUISITION SUMMARY')
print(f'  APTOS source     : {APTOS_MODE}  ->  {os.path.dirname(APTOS_CSV)}')
print(f'  EyePACS source   : {EYEPACS_MODE}  ->  {EYEPACS_DIR}')
print(f'  APTOS download   : {"PERFORMED" if APTOS_MODE == "KAGGLE API DOWNLOAD" else "SKIPPED"}')
print(f'  EyePACS download : {"PERFORMED" if EYEPACS_MODE == "KAGGLE API DOWNLOAD" else "SKIPPED"}')
print('EyePACS labels  :', LABELS_PATH)
print('EyePACS volumes :')
for v in VOLUMES:
    if v['kind'] == 'zip': print('   zip ', len(v['paths']), 'part(s):', [os.path.basename(p) for p in v['paths']])
    else: print('   dir ', v['root'])
print('APTOS labels    :', APTOS_CSV, f'({_best_n} rows)')
print('APTOS folder    :', sorted(os.listdir(os.path.dirname(APTOS_CSV))))

ARCHIVE_FINGERPRINT = [[os.path.basename(q), os.path.getsize(q), head_tail_md5(q)] for v in VOLUMES if v['kind'] == 'zip' for q in v['paths']] or [['dir-mode', sum(IMG_DIRS.get(v['root'], 0) for v in VOLUMES)]]

In [ ]:
def read_labels_raw():
    return read_csv_any(LABELS_PATH)

print('trainLabels:', LABELS_PATH, fmt_bytes(os.path.getsize(LABELS_PATH)))
_lab_raw = read_labels_raw()
assert {'image', 'level'} <= set(_lab_raw.columns), f'unexpected trainLabels columns: {list(_lab_raw.columns)}'
N_LABELED_EST = len(_lab_raw)
print('labelled rows:', N_LABELED_EST)

ZIP_ENTRY_COUNTS = []
RAW_EXTRACT_DIRS = []
for vi, v in enumerate(VOLUMES):
    if v['kind'] == 'zip':
        tot = sum(os.path.getsize(p) for p in v['paths'])
        print(f'volume {vi}: {len(v["paths"])} part(s), {fmt_bytes(tot)}')
        for p in v['paths']:
            print(f'    {os.path.basename(p):28s} {fmt_bytes(os.path.getsize(p))}')
        try:
            with open_zip(v['paths']) as z:
                infos = z.infolist()
                print(f'    -> valid zip, {len(infos)} entries, first: {[i.filename for i in infos[:3]]}')
                ZIP_ENTRY_COUNTS.append(len(infos))
        except (zipfile.BadZipFile, OSError, ValueError, EOFError) as e:
            print(f'    !! concatenated read failed ({e}); falling back to 7-Zip extraction')
            exe = shutil.which('7z') or shutil.which('7za') or shutil.which('7zr')
            if exe is None:
                raise RuntimeError(f'Volume {vi} is not a valid concatenated zip ({e}) and no 7z executable is available. '
                                   f'Install p7zip (apt-get install -y p7zip-full) and re-run.') from e
            dest = os.path.join(PATHS.cache_root, f'raw_volume_{vi}')
            subprocess.check_call([exe, 'x', v['paths'][0], f'-o{dest}', '-y', '-bso0', '-bsp0'])
            RAW_EXTRACT_DIRS.append(dest)
            VOLUMES[vi] = dict(kind='dir', root=dest)
            _n = 0
            for dp, dn, fn in os.walk(dest):
                for f in fn:
                    if f.lower().endswith(IMG_EXT):
                        IMG_INDEX.setdefault(os.path.splitext(f)[0], []).append(os.path.join(dp, f)); _n += 1
            print(f'    -> extracted {_n} images to {dest} (removed again after caching)')
    else:
        print(f'volume {vi}: directory {v["root"]} ({IMG_DIRS.get(v["root"], 0)} images)')
assert N_LABELED_EST > 0

In [ ]:
ARCHIVE_INDEX = {}
for vi, v in enumerate(VOLUMES):
    if v['kind'] == 'zip':
        with open_zip(v['paths']) as z:
            for info in z.infolist():
                if info.is_dir() or not info.filename.lower().endswith(IMG_EXT):
                    continue
                stem = os.path.splitext(os.path.basename(info.filename))[0]
                ARCHIVE_INDEX.setdefault(stem, (vi, info.filename, info.header_offset))
    else:
        for stem, paths in IMG_INDEX.items():
            for p in paths:
                if p.startswith(v['root']):
                    ARCHIVE_INDEX.setdefault(stem, (vi, p, 0)); break
print('images indexed in training volumes:', len(ARCHIVE_INDEX))
assert len(ARCHIVE_INDEX) > 0
LABELS_MD5 = file_md5(LABELS_PATH)
DATA_SIG = make_sig(LABELS_MD5, ARCHIVE_FINGERPRINT)
print('EyePACS dataset signature:', DATA_SIG)

In [ ]:
lab = read_labels_raw()
df = pd.DataFrame({'image_id': lab['image'].astype(str), 'label': lab['level'].astype(int)})
_pe = df.image_id.str.rsplit('_', n=1, expand=True)
df['patient_id'] = _pe[0]
df['eye'] = _pe[1]
assert df.image_id.is_unique, 'duplicate image ids in trainLabels'
assert set(df.eye.unique()) <= {'left', 'right'}, df.eye.unique()
assert set(df.label.unique()) <= set(range(CFG.num_classes)), df.label.unique()
df['in_archive'] = df.image_id.isin(ARCHIVE_INDEX)
df['src_vol'] = df.image_id.map(lambda s: ARCHIVE_INDEX[s][0] if s in ARCHIVE_INDEX else -1)
df['src_member'] = df.image_id.map(lambda s: ARCHIVE_INDEX[s][1] if s in ARCHIVE_INDEX else '')
df['src_order'] = df.image_id.map(lambda s: ARCHIVE_INDEX[s][2] if s in ARCHIVE_INDEX else -1)
print(df.head())

In [ ]:
n_missing = int((~df.in_archive).sum())
print('Dataset location        :', EYEPACS_DIR)
print('trainLabels.csv         :', LABELS_PATH)
print('Number of labelled imgs :', len(df))
print('Unique patients         :', df.patient_id.nunique())
print('Left / right images     :', int((df.eye == 'left').sum()), '/', int((df.eye == 'right').sum()))
print('Missing from archives   :', n_missing)
print('Class distribution      :')
print(df.label.value_counts().sort_index().rename('n').to_frame().assign(pct=lambda d: (100 * d.n / d.n.sum()).round(2)))
EXPECTED_EYEPACS_TRAIN = 35126          # documented size of the labelled Kaggle EyePACS training set
if len(df) >= 0.99 * EXPECTED_EYEPACS_TRAIN:
    print(f'full labelled EyePACS training set ({len(df)} vs {EXPECTED_EYEPACS_TRAIN} documented)')
else:
    print(f'WARNING: {len(df)} labelled images vs {EXPECTED_EYEPACS_TRAIN} documented -> PARTIAL or non-standard attachment; results will not be comparable to the full set.')
if n_missing / len(df) > CFG.max_missing_frac:
    raise RuntimeError(f'{n_missing} labelled images ({100*n_missing/len(df):.2f}%) are missing from the archives - refusing to continue.')
if n_missing:
    print(f'WARNING: dropping {n_missing} labelled images that are absent from the archives:', df.loc[~df.in_archive, 'image_id'].head(10).tolist(), '...')
    df = df[df.in_archive].reset_index(drop=True)
print('\nNOTE on official splits: the Kaggle competition provides labels only for the training images (test labels are not public),')
print('so a patient-safe subset is selected from the labelled set (Cell 10A) and that subset is split by patient with class stratification (Cell 11).')

In [ ]:
EYE_SUBSET_SIZE = CFG.eye_subset_size
EYE_SUBSET_TAG = f'{EYE_SUBSET_SIZE // 1000}k' if EYE_SUBSET_SIZE % 1000 == 0 else str(EYE_SUBSET_SIZE)
SUBSET_SEED = CFG.subset_seed
df_full = df.copy()
NC = CFG.num_classes
FULL_DIST = df_full.label.value_counts().reindex(range(NC)).fillna(0).astype(int)

def subset_targets(avail, n_target, min_frac):
    floor = np.minimum(avail, int(math.ceil(min_frac * n_target)))
    fixed = np.zeros(len(avail), bool)
    for _ in range(len(avail) + 1):
        free = ~fixed
        t = np.zeros(len(avail)); t[fixed] = floor[fixed]
        if free.any():
            t[free] = avail[free] / avail[free].sum() * (n_target - floor[fixed].sum())
        low = free & (t < floor)
        if not low.any(): break
        fixed |= low
    t = np.minimum(np.round(t).astype(int), avail)
    t[np.argmax(t)] += n_target - t.sum()
    return t

def select_patient_subset(frame, n_target, seed, min_frac):
    avail = frame.label.value_counts().reindex(range(NC)).fillna(0).astype(int).values
    n_target = min(n_target, len(frame))
    target = subset_targets(avail, n_target, min_frac)
    counts = pd.crosstab(frame.patient_id, frame.label).reindex(columns=range(NC), fill_value=0)
    pids, P = counts.index.to_numpy(), counts.to_numpy()
    sizes = P.sum(1)
    rarity = avail / avail.sum()
    key = np.where(P > 0, rarity[None, :], np.inf).min(1)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(pids)); order = order[np.argsort(key[order], kind='stable')]
    chosen, cur = np.zeros(len(pids), bool), np.zeros(NC, int)
    for i in order:
        if (cur + P[i] <= target).all():
            chosen[i] = True; cur += P[i]
    for tol in (1.03, 1.10, None):
        remaining = n_target - int(cur.sum())
        if remaining <= 0: break
        for i in order:
            if chosen[i] or sizes[i] > remaining: continue
            if tol is not None and not (cur + P[i] <= np.ceil(target * tol)).all(): continue
            chosen[i] = True; cur += P[i]; remaining -= int(sizes[i])
            if remaining <= 0: break
    return pids[chosen], target, cur

SUBSET_PIDS, SUBSET_TARGET, SUBSET_ACHIEVED = select_patient_subset(df_full, EYE_SUBSET_SIZE, SUBSET_SEED, CFG.subset_min_class_frac)
df = df_full[df_full.patient_id.isin(set(SUBSET_PIDS))].sort_values('image_id').reset_index(drop=True)

SUB_DIST = df.label.value_counts().reindex(range(NC)).fillna(0).astype(int)
print(f'EyePACS full dataset : {len(df_full):,} labelled images, {df_full.patient_id.nunique():,} patients')
print(f'Requested subset     : {EYE_SUBSET_SIZE:,} images | seed {SUBSET_SEED} | grade floor {100 * CFG.subset_min_class_frac:.1f}% of the subset')
print(f'Selected subset      : {len(df):,} images, {df.patient_id.nunique():,} patients')
_n_eyes = df.groupby('patient_id').size()
print(f'  patients with both eyes: {int((_n_eyes == 2).sum()):,} | with a single eye: {int((_n_eyes == 1).sum()):,}')
print(pd.DataFrame({'grade': range(NC), 'full_n': FULL_DIST.values, 'full_%': (100 * FULL_DIST / FULL_DIST.sum()).round(2).values, 'target_n': SUBSET_TARGET,
                    'subset_n': SUB_DIST.values, 'subset_%': (100 * SUB_DIST / SUB_DIST.sum()).round(2).values}).to_string(index=False))
if len(df) != EYE_SUBSET_SIZE:
    print(f'NOTE: {len(df):,} images were selected instead of exactly {EYE_SUBSET_SIZE:,}. Both eyes of a patient are never separated, so the closest reachable size '
          f'was used (difference {len(df) - EYE_SUBSET_SIZE:+d}).' + (' The labelled set itself has fewer images than requested.' if len(df_full) < EYE_SUBSET_SIZE else ''))
if (SUB_DIST / SUB_DIST.sum()).max() > 0.85:
    print('WARNING: severe class imbalance in the selected subset.')

assert df.image_id.is_unique, 'duplicate image ids in the subset'
assert (SUB_DIST > 0).all(), 'a DR grade is missing from the subset'
_sel_sizes = df.groupby('patient_id').size()
assert (_sel_sizes == df_full.groupby('patient_id').size().reindex(_sel_sizes.index)).all(), 'a selected patient lost an eye (patients must stay whole)'
assert len(df) >= 0.99 * min(EYE_SUBSET_SIZE, len(df_full)), f'subset size {len(df)} is too far from the requested {EYE_SUBSET_SIZE}'
SUBSET_INFO = dict(requested=EYE_SUBSET_SIZE, selected=len(df), patients=int(df.patient_id.nunique()), seed=SUBSET_SEED, min_class_frac=CFG.subset_min_class_frac,
                   full_images=len(df_full), full_patients=int(df_full.patient_id.nunique()), target_per_grade=[int(x) for x in SUBSET_TARGET],
                   achieved_per_grade=[int(x) for x in SUB_DIST.values], full_per_grade=[int(x) for x in FULL_DIST.values])

In [ ]:
f_tr, f_va, f_te = CFG.split_fracs
assert abs(f_tr + f_va + f_te - 1) < 1e-9
N_FOLDS = 10
n_va, n_te = int(round(f_va * N_FOLDS)), int(round(f_te * N_FOLDS))
assert abs(n_va / N_FOLDS - f_va) < 1e-9 and abs(n_te / N_FOLDS - f_te) < 1e-9, 'split fractions must be multiples of 0.1'
fold = np.full(len(df), -1)
for k, (_, idx) in enumerate(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=CFG.seed).split(df, df.label, groups=df.patient_id)):
    fold[idx] = k
df['split'] = np.where(fold < n_va, 'val', np.where(fold < n_va + n_te, 'test', 'train'))
assert df.split.notna().all()

In [ ]:
tr_p, va_p, te_p = (set(df.loc[df.split == s, 'patient_id']) for s in ('train', 'val', 'test'))
assert not (tr_p & va_p), 'patient leakage train/val'
assert not (tr_p & te_p), 'patient leakage train/test'
assert not (va_p & te_p), 'patient leakage val/test'
assert len(tr_p) + len(va_p) + len(te_p) == df.patient_id.nunique()
assert df.image_id.is_unique
for s in ('train', 'val', 'test'):
    print(f'{s:5s} images {int((df.split == s).sum()):6d} | patients {df.loc[df.split == s, "patient_id"].nunique():6d}')
dist = pd.crosstab(df.split, df.label).loc[['train', 'val', 'test']]
print('\nClass distribution per split (images):'); print(dist)
print('\nClass distribution per split (%):'); print((100 * dist.div(dist.sum(1), axis=0)).round(2))
print('\nNo patient appears in more than one split. OK')

manifest = df[['image_id', 'patient_id', 'eye', 'label']].copy()
manifest['filename'] = df.src_member.map(os.path.basename)
manifest['split'] = df.split
manifest = manifest.sort_values('image_id').reset_index(drop=True)
MANIFEST_PATH = os.path.join(PATHS.out, f'eyepacs_{EYE_SUBSET_TAG}_manifest.csv')
manifest.to_csv(MANIFEST_PATH, index=False)
MANIFEST_HASH = hashlib.md5(manifest.to_csv(index=False).encode()).hexdigest()
json.dump(dict(manifest_md5=MANIFEST_HASH, split_fracs=CFG.split_fracs, labels_md5=LABELS_MD5, data_signature=DATA_SIG, **SUBSET_INFO,
               selection=('whole patients; per-grade targets proportional to the full EyePACS distribution with a floor of subset_min_class_frac per grade; seeded random order, '
                          'rarest-grade patients first, fit-within-target pass then smallest-overshoot fill; split by StratifiedGroupKFold on patient ids')),
          open(MANIFEST_PATH.replace('.csv', '_meta.json'), 'w'), indent=1)
PATHS.cache_eyepacs = os.path.join(PATHS.cache_root, f'eyepacs_{CFG.img_size}_{PRE_SIG}_{DATA_SIG}_{EYE_SUBSET_TAG}_{MANIFEST_HASH[:8]}')
os.makedirs(PATHS.cache_eyepacs, exist_ok=True)
df['image_path'] = PATHS.cache_eyepacs + '/' + df.image_id + '.jpg'
print(f'subset manifest : {MANIFEST_PATH}  (md5 {MANIFEST_HASH})')
print('EyePACS cache   :', PATHS.cache_eyepacs)

def dist_report(named):
    cnt = pd.DataFrame({n: f.label.value_counts() for n, f in named.items()}).reindex(range(CFG.num_classes)).fillna(0).astype(int).T
    print('counts:'); print(cnt.to_string()); print('percent:'); print((100 * cnt.div(cnt.sum(1), axis=0)).round(2).to_string())
print('\nClass distribution: full EyePACS / selected subset / train / validation / test')
dist_report({'full EyePACS': df_full, 'selected subset': df, 'train': df[df.split == 'train'], 'val': df[df.split == 'val'], 'test': df[df.split == 'test']})

In [ ]:
def crop_fundus(img_bgr):
    h, w = img_bgr.shape[:2]
    gray = cv2.medianBlur(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), 5)
    mask = gray > CFG.crop_threshold
    rows, cols = np.flatnonzero(mask.any(1)), np.flatnonzero(mask.any(0))
    if rows.size == 0 or cols.size == 0:
        return img_bgr, False
    y0, y1, x0, x1 = rows[0], rows[-1] + 1, cols[0], cols[-1] + 1
    if (y1 - y0) * (x1 - x0) < 0.10 * h * w:
        return img_bgr, False
    return img_bgr[y0:y1, x0:x1], True

def pad_to_square(img):
    h, w = img.shape[:2]; s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    return cv2.copyMakeBorder(img, top, s - h - top, left, s - w - left, cv2.BORDER_CONSTANT, value=(0, 0, 0))

def resize_square(img, size):
    interp = cv2.INTER_AREA if img.shape[0] >= size else cv2.INTER_CUBIC
    return cv2.resize(img, (size, size), interpolation=interp)

def ben_graham(img, sigma):
    return cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), sigma), -4, 128)

def preprocess_array(img_bgr):
    img, found = crop_fundus(img_bgr)
    img = resize_square(pad_to_square(img), CFG.img_size)
    return ben_graham(img, CFG.img_size * CFG.bg_sigma_frac), found

def preprocess_bytes(data):
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError('cv2.imdecode returned None (corrupt/unsupported image)')
    return preprocess_array(img)

In [ ]:
import multiprocessing as mp
_VOL_HANDLES = {}

def _read_source(vi, member):
    if vi < 0:
        with open(member, 'rb') as f:
            return f.read()
    v = VOLUMES[vi]
    if v['kind'] == 'dir':
        with open(member, 'rb') as f:
            return f.read()
    key = (os.getpid(), vi)
    if key not in _VOL_HANDLES:
        _VOL_HANDLES[key] = open_zip(v['paths'])
    return _VOL_HANDLES[key].read(member)

def _cache_task(task):
    out = []
    for image_id, vi, member, out_path in task:
        try:
            if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
                out.append((image_id, 'cached', '')); continue
            img, found = preprocess_bytes(_read_source(vi, member))
            ok = cv2.imwrite(out_path, img, [int(cv2.IMWRITE_JPEG_QUALITY), CFG.jpeg_quality])
            if not ok:
                raise IOError('cv2.imwrite failed')
            out.append((image_id, 'ok' if found else 'ok_nocrop', ''))
        except Exception as e:
            out.append((image_id, 'fail', f'{type(e).__name__}: {e}'))
    return out

def build_cache(items, name, chunk=48):
    tasks = [items[i:i + chunk] for i in range(0, len(items), chunk)]
    n_proc = max(1, min(CFG.max_cache_procs, USABLE_CPUS))
    status, fails, done, t0, next_report = Counter(), [], 0, time.time(), 0.05
    print(f'[{name}] caching {len(items)} images with {n_proc} processes -> {os.path.dirname(items[0][3])}')
    with mp.get_context('fork').Pool(n_proc) as pool:
        for res in pool.imap_unordered(_cache_task, tasks):
            for image_id, st, msg in res:
                status[st] += 1; done += 1
                if st == 'fail':
                    fails.append((image_id, msg))
                    if len(fails) <= 20:
                        print(f'  FAILED {image_id}: {msg}')
            if done / len(items) >= next_report:
                el = time.time() - t0
                print(f'  {done}/{len(items)} ({100*done/len(items):.0f}%)  {el/60:.1f} min elapsed, ETA {el/done*(len(items)-done)/60:.1f} min  fails={len(fails)}')
                next_report += 0.05
    return status, fails

def cache_report(name, status, fails, directory, n_items):
    size = sum(e.stat().st_size for e in os.scandir(directory) if e.is_file())
    print(f'--- {name} cache report ---')
    print('Number of images processed        :', n_items)
    print('Successfully cached (new)         :', status['ok'] + status['ok_nocrop'])
    print('  already cached (resumed)        :', status['cached'])
    print('  crop fallback (full frame used) :', status['ok_nocrop'])
    print('Failed                            :', len(fails))
    print('Cache directory                   :', directory)
    print('Approximate cache size            :', fmt_bytes(size))
    if fails:
        fpath = os.path.join(PATHS.out, f'preprocess_failures_{name}.csv')
        pd.DataFrame(fails, columns=['image_id', 'error']).to_csv(fpath, index=False)
        print('Failure log                       :', fpath)

def measure_cache_kb(n=24):
    rows = df.sort_values(['src_vol', 'src_order']); step = max(1, len(rows) // n); sizes = []; t0 = time.time()
    for r in rows.iloc[::step].head(n).itertuples():
        img, _ = preprocess_bytes(_read_source(int(r.src_vol), r.src_member))
        sizes.append(len(cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), CFG.jpeg_quality])[1]))
    return float(np.mean(sizes)) / 1024, (time.time() - t0) / max(1, len(sizes))

_kb, _spi = measure_cache_kb()
print(f'measured cached JPEG size: {_kb:.1f} KB/image')
EST_CACHE_BYTES = int(len(df) * _kb * 1024)
EST_PRE_S = _spi * len(df) / max(1, min(CFG.max_cache_procs, USABLE_CPUS))
print(f'estimated EyePACS preprocessing time: ~{EST_PRE_S / 60:.1f} min for the {len(df):,} SELECTED images (measured {_spi:.2f} s/image on {24} samples / {min(CFG.max_cache_procs, USABLE_CPUS)} processes; the actual time is printed afterwards)')
json.dump(dict(signature=PRE_SIG, version=CFG.pre_version, size=CFG.img_size, crop_threshold=CFG.crop_threshold, bg_sigma_frac=CFG.bg_sigma_frac,
               jpeg_quality=CFG.jpeg_quality, data_signature=DATA_SIG, labels_md5=LABELS_MD5, archives=ARCHIVE_FINGERPRINT, manifest_hash=MANIFEST_HASH, subset_size=EYE_SUBSET_SIZE, subset_seed=SUBSET_SEED), open(os.path.join(PATHS.cache_eyepacs, 'cache_meta.json'), 'w'))

_items = [(r.image_id, int(r.src_vol), r.src_member, r.image_path) for r in df.sort_values(['src_vol', 'src_order']).itertuples()]
_t0 = time.time()
_status, _fails = build_cache(_items, 'eyepacs')
cache_report('eyepacs', _status, _fails, PATHS.cache_eyepacs, len(_items))
print(f'EyePACS caching took {(time.time()-_t0)/60:.1f} min')
TIMES['preprocess_eyepacs_s'] = time.time() - _t0

df['cached'] = [os.path.exists(p) and os.path.getsize(p) > 0 for p in df.image_path]
_n_fail = int((~df.cached).sum())
if _n_fail / len(df) > CFG.max_fail_frac:
    raise RuntimeError(f'{_n_fail} images ({100*_n_fail/len(df):.2f}%) failed preprocessing - too many to drop silently. See the failure log.')
if _n_fail:
    print(f'WARNING: {_n_fail} images failed and are excluded (patient assignment unchanged, so no leakage):', df.loc[~df.cached, 'image_id'].tolist()[:20])
if _n_fail:
    _lost = df.loc[~df.cached]; _pat_lost_any = _lost.patient_id.nunique()
    _pat_lost_all = int((df.groupby('patient_id').cached.sum() == 0).sum())
    print(f'patients that lost at least one eye: {_pat_lost_any} | patients that lost every eye: {_pat_lost_all}')
df_selected = df.copy()
df_all_labelled = df_selected
df = df[df.cached].reset_index(drop=True)
for _d in RAW_EXTRACT_DIRS:
    shutil.rmtree(_d, ignore_errors=True)
if RAW_EXTRACT_DIRS: print('removed raw 7-Zip extraction dirs:', RAW_EXTRACT_DIRS)

_rng = np.random.default_rng(CFG.seed)
for _p in df.image_path.sample(25, random_state=CFG.seed):
    _im = cv2.imread(_p, cv2.IMREAD_COLOR)
    assert _im is not None and _im.shape == (CFG.img_size, CFG.img_size, 3), _p
print(f'full EyePACS: {len(df_full):,} | selected subset: {len(df_selected):,} | preprocessed successfully: {len(df):,} | failed/excluded: {len(df_selected) - len(df)}')
_ne = df.groupby('patient_id').size()
print(f'patients represented: {df.patient_id.nunique():,} | with both eyes usable: {int((_ne == 2).sum()):,} | with only one usable eye: {int((_ne == 1).sum()):,}')
for s in ('train', 'val', 'test'):
    print(f'{s:5s}: {int((df.split == s).sum()):6d} images ({100 * (df.split == s).mean():.1f}% of usable) | {df.loc[df.split == s, "patient_id"].nunique():6d} patients')
print('post-cache class distribution per split (% of images) - check that failures did not remove rare grades disproportionately:')
print((100 * pd.crosstab(df.split, df.label, normalize='index').loc[['train', 'val', 'test']]).round(2))
assert not (set(df.loc[df.split == 'train', 'patient_id']) & set(df.loc[df.split == 'test', 'patient_id']))
assert not (set(df.loc[df.split == 'train', 'patient_id']) & set(df.loc[df.split == 'val', 'patient_id']))
assert not (set(df.loc[df.split == 'val', 'patient_id']) & set(df.loc[df.split == 'test', 'patient_id']))

In [ ]:
_checks = []
def _check(label, cond, detail=''):
    _checks.append((label, bool(cond))); print(f'  [{"OK" if cond else "FAIL"}] {label} {detail}')

print('PRE-TRAINING SANITY CHECKS')
_p = {s: set(df.loc[df.split == s, 'patient_id']) for s in ('train', 'val', 'test')}
_check('selected images are close to the requested subset size', len(df_selected) >= 0.99 * min(EYE_SUBSET_SIZE, len(df_full)),
       f'({len(df_selected):,} selected vs {EYE_SUBSET_SIZE:,} requested; {len(df):,} usable after preprocessing)')
_check('>= 97% of the selected images were preprocessed successfully', len(df) >= 0.97 * len(df_selected), f'({len(df):,}/{len(df_selected):,})')
_check('no duplicate image ids', df.image_id.is_unique)
_check('no patient in more than one split (train/val, train/test, val/test)', not (_p['train'] & _p['val']) and not (_p['train'] & _p['test']) and not (_p['val'] & _p['test']))
_check('all labels are grades 0-4', set(df.label.unique()) <= set(range(NC)))
_check('all selected (cached) files exist', bool(df.image_path.map(lambda q: os.path.exists(q) and os.path.getsize(q) > 0).all()))
_dec = True
for _s in ('train', 'val', 'test'):
    for _q in df.loc[df.split == _s, 'image_path'].sample(8, random_state=CFG.seed):
        _im = cv2.imread(_q, cv2.IMREAD_COLOR); _dec = _dec and _im is not None and _im.shape == (CFG.img_size, CFG.img_size, 3)
_check('sample images decode (8 per split)', _dec)
DIST_CNT = pd.DataFrame({'full EyePACS': df_full.label.value_counts(), 'selected subset': df_selected.label.value_counts(), 'train': df.loc[df.split == 'train', 'label'].value_counts(),
                         'val': df.loc[df.split == 'val', 'label'].value_counts(), 'test': df.loc[df.split == 'test', 'label'].value_counts()}).reindex(range(NC)).fillna(0).astype(int).T
DIST_PCT = (100 * DIST_CNT.div(DIST_CNT.sum(1), axis=0)).round(2)
_check('every DR grade is present in train, validation and test', bool((DIST_CNT.loc[['train', 'val', 'test']] > 0).all().all()))
_meta = json.load(open(os.path.join(PATHS.cache_eyepacs, 'cache_meta.json')))
_check('cache signature includes the subset manifest hash', MANIFEST_HASH[:8] in os.path.basename(PATHS.cache_eyepacs) and _meta.get('manifest_hash') == MANIFEST_HASH)
_check('APTOS is kept separate from the EyePACS data (no overlap of folders, cache not yet created)', not APTOS_CSV.startswith(EYEPACS_DIR) and PATHS.cache_aptos is None)
print(f'  split sizes (usable images): train {int((df.split == "train").sum()):,} | validation {int((df.split == "val").sum()):,} | test {int((df.split == "test").sum()):,}')
print(f'  manifest: {MANIFEST_PATH}  md5 {MANIFEST_HASH}')
print('\nClass distribution (counts):'); print(DIST_CNT.to_string())
print('\nClass distribution (%):'); print(DIST_PCT.to_string())
_bad = [l for l, ok in _checks if not ok]
if _bad:
    raise RuntimeError('Sanity checks failed - stopping before training: ' + '; '.join(_bad))
print('\nAll sanity checks passed.')

_cache_bytes = sum(e.stat().st_size for e in os.scandir(PATHS.cache_eyepacs) if e.is_file())
_full_train = 0.7 * len(df_full)
print('\nRUNTIME PLAN')
print(f'  EyePACS full dataset            : {len(df_full):,} images')
print(f'  Selected subset                 : {len(df_selected):,} images ({len(df):,} usable)  -> train {int((df.split == "train").sum()):,} | validation {int((df.split == "val").sum()):,} | test {int((df.split == "test").sum()):,}')
print(f'  APTOS (external, labelled rows) : {_best_n:,}')
print(f'  EyePACS preprocessing           : estimated {EST_PRE_S / 60:.1f} min before caching -> actual {TIMES["preprocess_eyepacs_s"] / 60:.1f} min (only the selected images were decoded)')
print(f'  EyePACS cache size              : estimated {fmt_bytes(EST_CACHE_BYTES)} -> actual {fmt_bytes(_cache_bytes)}')
print(f'  Training cost                   : scales ~linearly with the number of training images = {int((df.split == "train").sum()) / _full_train:.2f}x the cost of a 70% split of the full dataset;')
print('                                    the absolute time per model is measured during its first epoch and printed (ETA) in Cells 24-26 - it is not guessed here')
print('  Final actual timings (download, preprocessing, per-model training, ensemble, inference, total wall-clock) are printed in Cell 51.')
if CFG.delete_eyepacs_archives_after_cache and LABELS_PATH.startswith(PATHS.eyepacs_dl):
    shutil.rmtree(PATHS.eyepacs_dl, ignore_errors=True)
    print('removed the downloaded EyePACS archives (CFG.delete_eyepacs_archives_after_cache=True); a rerun in a fresh session would download them again')

In [ ]:
DF = {s: df[df.split == s].reset_index(drop=True) for s in ('train', 'val', 'test')}
_cpu = USABLE_CPUS
W_TRAIN = min(6, max(2, _cpu // 2) if PARALLEL else max(2, _cpu - 1))
W_EVAL = min(4, max(2, _cpu // 2))

class CachedFundus(Dataset):
    def __init__(self, frame, label_col='label'):
        self.paths = frame.image_path.tolist()
        self.labels = frame[label_col].values.astype(np.int64)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = cv2.imread(self.paths[i], cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f'could not read cached image {self.paths[i]}')
        return torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), int(self.labels[i]), i

def make_loader(frame, train, workers=None, seed=0):
    workers = (W_TRAIN if train else W_EVAL) if workers is None else workers
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(CachedFundus(frame), batch_size=CFG.batch_size if train else CFG.eval_batch_size,
                      shuffle=train, drop_last=False, num_workers=workers, pin_memory=True,
                      worker_init_fn=seed_worker, generator=g, persistent_workers=bool(train and workers > 0),
                      prefetch_factor=2 if workers > 0 else None)
print({k: len(v) for k, v in DF.items()}, '| train workers', W_TRAIN, '| eval workers', W_EVAL)

In [ ]:
def to_float(xb_uint8_nhwc, dev):
    return xb_uint8_nhwc.to(dev, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)

def gpu_augment(x, gen):
    N, _, H, W = x.shape; dev = x.device
    def U(lo, hi): return lo + (hi - lo) * torch.rand(N, device=dev, generator=gen)
    b = (1 + U(-CFG.aug_brightness, CFG.aug_brightness)).view(N, 1, 1, 1)
    c = (1 + U(-CFG.aug_contrast, CFG.aug_contrast)).view(N, 1, 1, 1)
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    x = (((x - m) * c + m) * b).clamp_(0, 1)
    ang = U(*CFG.aug_rotation_range_deg) * math.pi / 180
    sx = 1.0 - 2.0 * (torch.rand(N, device=dev, generator=gen) < 0.5).float()
    sy = 1.0 - 2.0 * (torch.rand(N, device=dev, generator=gen) < 0.5).float()
    area = U(*CFG.aug_crop_scale)
    ratio = torch.exp(U(-CFG.aug_crop_ratio_jitter, CFG.aug_crop_ratio_jitter))
    fx = (area.sqrt() * ratio.sqrt()).clamp(max=1.0); fy = (area.sqrt() / ratio.sqrt()).clamp(max=1.0)
    tx = U(-1, 1) * (1 - fx); ty = U(-1, 1) * (1 - fy)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(N, 2, 3, device=dev)
    theta[:, 0, 0] = cos * fx * sx; theta[:, 0, 1] = -sin * fy * sy; theta[:, 0, 2] = tx
    theta[:, 1, 0] = sin * fx * sx; theta[:, 1, 1] = cos * fy * sy;  theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x - 0.5, grid, mode='bilinear', padding_mode='zeros', align_corners=False) + 0.5

def normalize(x, model):
    return (x - model.dr_mean) / model.dr_std

In [ ]:
_cnt = DF['train'].label.value_counts().reindex(range(CFG.num_classes)).fillna(0).astype(int)
assert (_cnt > 0).all(), 'a class is missing from the training split'
_freq = _cnt / _cnt.sum()
_raw = 1.0 / np.sqrt(_freq)
_w = _raw / (_raw * _freq).sum()
CLASS_W = torch.tensor(_w.values, dtype=torch.float32)
print(pd.DataFrame({'class': range(5), 'name': CFG.class_names, 'count': _cnt.values, 'raw_frequency': _freq.values.round(5), 'weight': _w.values.round(4)}).to_string(index=False))
assert abs(float((CLASS_W * torch.tensor(_freq.values, dtype=torch.float32)).sum()) - 1) < 1e-4

In [ ]:
# ===== CELL 18 — MODEL IMPORTS / timm + HUGGING FACE HUB SETUP =====
RULES = {
    'resnet50': dict(
        frozen=['conv1.', 'bn1.', 'layer1.'],
        trainable=['layer2.', 'layer3.', 'layer4.', 'fc.'], head=['fc.']),
    'efficientnet_b0': dict(
        frozen=['conv_stem.', 'bn1.', 'blocks.0.', 'blocks.1.', 'blocks.2.'],
        trainable=['blocks.3.', 'blocks.4.', 'blocks.5.', 'blocks.6.', 'conv_head.', 'bn2.', 'classifier.'], head=['classifier.']),
    'vit_tiny': dict(
        frozen=['cls_token', 'pos_embed', 'patch_embed.'] + [f'blocks.{i}.' for i in range(0, 4)],
        trainable=[f'blocks.{i}.' for i in range(4, 12)] + ['norm.', 'head.'], head=['head.']),
}

for _k, _v in MODELS.items():
    if not timm.list_pretrained(_v['timm_id']):
        raise RuntimeError(f'{_v["timm_id"]} is not a pretrained model in timm {timm.__version__}; upgrade timm (pip install -U timm).')

def build_model(key, pretrained=True):
    spec, rules = MODELS[key], RULES[key]
    try:
        m = timm.create_model(spec['timm_id'], pretrained=pretrained, num_classes=CFG.num_classes)
    except Exception as e:
        raise RuntimeError(f'Could not load pretrained weights for {spec["timm_id"]} - is Internet enabled in the notebook settings? ({e})') from e
    pc = m.pretrained_cfg
    m.register_buffer('dr_mean', torch.tensor(pc['mean'], dtype=torch.float32).view(1, 3, 1, 1))
    m.register_buffer('dr_std', torch.tensor(pc['std'], dtype=torch.float32).view(1, 3, 1, 1))
    for n, p in m.named_parameters():
        fro = any(n.startswith(s) for s in rules['frozen'])
        trn = any(n.startswith(s) for s in rules['trainable'])
        if fro == trn:
            raise RuntimeError(f'{key}: parameter {n} is {"in both" if fro else "in neither"} the frozen/trainable rules')
        p.requires_grad = trn
    frozen_prefixes = [s for s in rules['frozen'] if s.endswith('.')]
    m.frozen_module_names = {name for name, _ in m.named_modules() if name and any((name + '.').startswith(s) for s in frozen_prefixes)}
    m._cl = spec['channels_last']
    return m

def set_train_mode(model):
    model.train()
    for name, mod in model.named_modules():
        if name in model.frozen_module_names:
            mod.eval()

def make_optimizer(model, key):
    head_pref = RULES[key]['head']
    head = [p for n, p in model.named_parameters() if p.requires_grad and any(n.startswith(h) for h in head_pref)]
    body = [p for n, p in model.named_parameters() if p.requires_grad and not any(n.startswith(h) for h in head_pref)]
    groups = [dict(params=head, lr=CFG.lr_head, name='head')]
    if body:
        groups.insert(0, dict(params=body, lr=CFG.lr_backbone, name='backbone'))
    return torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)
print('timm pretrained identifiers verified:', [v['timm_id'] for v in MODELS.values()])

In [ ]:
def build_resnet50():
    return build_model('resnet50')

VERIFY_MODELS = OrderedDict()
VERIFY_MODELS['resnet50'] = build_resnet50()
print('ResNet50 built from', MODELS['resnet50']['timm_id'], '| frozen modules:', sorted(n for n in VERIFY_MODELS['resnet50'].frozen_module_names if '.' not in n))

In [ ]:
def build_efficientnet_b0():
    return build_model('efficientnet_b0')

VERIFY_MODELS['efficientnet_b0'] = build_efficientnet_b0()
print('EfficientNet-B0 built from', MODELS['efficientnet_b0']['timm_id'], '| stages in model.blocks:', len(VERIFY_MODELS['efficientnet_b0'].blocks))
assert len(VERIFY_MODELS['efficientnet_b0'].blocks) == 7

In [ ]:
def build_vit_tiny():
    return build_model('vit_tiny')

VERIFY_MODELS['vit_tiny'] = build_vit_tiny()
print('ViT-Tiny built from', MODELS['vit_tiny']['timm_id'], '| depth:', len(VERIFY_MODELS['vit_tiny'].blocks), '| img_size:', VERIFY_MODELS['vit_tiny'].patch_embed.img_size)
assert len(VERIFY_MODELS['vit_tiny'].blocks) == 12

In [ ]:
_rows = []
for key, m in VERIFY_MODELS.items():
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    rules = RULES[key]
    for n, p in m.named_parameters():
        if any(n.startswith(s) for s in rules['frozen']):
            assert not p.requires_grad, f'{key}: {n} should be frozen'
        if any(n.startswith(s) for s in rules['trainable']):
            assert p.requires_grad, f'{key}: {n} should be trainable'
    if trainable == total or trainable == 0:
        raise RuntimeError(f'{key}: freezing failed (trainable={trainable}, total={total}) - the whole model would be (un)trained')
    out_dim = m.get_classifier().out_features
    assert out_dim == CFG.num_classes, (key, out_dim)
    set_train_mode(m)
    n_frozen_bn = 0
    for name, mod in m.named_modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm) and name in m.frozen_module_names:
            assert not mod.training, f'{key}: frozen BN {name} is in train mode'; n_frozen_bn += 1
        elif isinstance(mod, nn.modules.batchnorm._BatchNorm):
            assert mod.training, f'{key}: trainable BN {name} should be in train mode'
    m.eval().to('cuda:0')
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=CFG.amp):
        y = m(normalize(torch.rand(2, 3, CFG.img_size, CFG.img_size, device='cuda:0'), m))
    assert tuple(y.shape) == (2, CFG.num_classes), y.shape
    assert torch.isfinite(y.float()).all()
    m.cpu()
    _rows.append(dict(model=MODELS[key]['display'], total_params=total, trainable=trainable, frozen=total - trainable,
                      trainable_pct=round(100 * trainable / total, 2), classifier_out=out_dim,
                      input=f'{CFG.img_size}x{CFG.img_size}x3', frozen_BN_layers=n_frozen_bn, output_shape=tuple(y.shape)))
print(pd.DataFrame(_rows).to_string(index=False))
del VERIFY_MODELS, m; gc.collect(); torch.cuda.empty_cache()
print('Model verification passed.')

In [ ]:
OOM_ERRORS = (torch.cuda.OutOfMemoryError,)
_LOG_LOCK = threading.Lock()
def log(tag, msg):
    with _LOG_LOCK:
        print(f'[{tag}] {msg}', flush=True)

def qwk_score(y, p):
    k = cohen_kappa_score(y, p, weights='quadratic', labels=list(range(CFG.num_classes)))
    return 0.0 if (k is None or not np.isfinite(k)) else float(k)

class WarmupCosine:
    def __init__(self, optimizer, warmup_steps, total_steps, min_factor):
        self.opt, self.warm, self.total, self.minf = optimizer, max(1, warmup_steps), total_steps, min_factor
        self.base = [g['lr'] for g in optimizer.param_groups]; self.n = 0; self.scale = 1.0
        self._apply()
    def factor(self, n):
        if n < self.warm: return (n + 1) / self.warm
        prog = min(1.0, (n - self.warm) / max(1, self.total - self.warm))
        return self.minf + (1 - self.minf) * 0.5 * (1 + math.cos(math.pi * prog))
    def _apply(self):
        for g, b in zip(self.opt.param_groups, self.base):
            g['lr'] = b * self.factor(self.n) * self.scale
    def step(self):
        self.n += 1; self._apply()
    def state_dict(self): return dict(n=self.n, scale=self.scale)
    def load_state_dict(self, d): self.n, self.scale = d['n'], d['scale']; self._apply()

def cpu_clone(obj):
    if torch.is_tensor(obj): return obj.detach().cpu().clone()
    if isinstance(obj, dict): return {k: cpu_clone(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return type(obj)(cpu_clone(v) for v in obj)
    return copy.deepcopy(obj)

def take_snapshot(model, optimizer, scaler, scheduler):
    return dict(model=cpu_clone(model.state_dict()), opt=cpu_clone(optimizer.state_dict()),
                scaler=copy.deepcopy(scaler.state_dict()), sched=scheduler.state_dict())

def restore_snapshot(model, optimizer, scaler, scheduler, snap):
    model.load_state_dict(snap['model']); optimizer.load_state_dict(copy.deepcopy(snap['opt']))
    scaler.load_state_dict(copy.deepcopy(snap['scaler'])); scheduler.load_state_dict(snap['sched'])

def repair_buffers(model, backup_state):
    fixed = 0
    for name, buf in model.named_buffers():
        if buf.is_floating_point() and not torch.isfinite(buf).all():
            buf.copy_(backup_state[name].to(buf.device)); fixed += 1
    return fixed

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=CFG.amp)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=CFG.amp)

@torch.no_grad()
def predict_logits(model, loader, dev, chunk=None):
    chunk = chunk or CFG.eval_chunk
    model.eval()
    outs, ys, idx = [], [], []
    for xb, yb, ib in loader:
        x = normalize(to_float(xb, dev), model)
        while True:
            try:
                parts = []
                for xc in x.split(chunk):
                    if model._cl: xc = xc.contiguous(memory_format=torch.channels_last)
                    with torch.autocast('cuda', dtype=torch.float16, enabled=CFG.amp):
                        parts.append(model(xc).float())
                logits = torch.cat(parts); break
            except OOM_ERRORS:
                parts = None; gc.collect(); torch.cuda.empty_cache(); chunk //= 2
                if chunk < 1: raise
        outs.append(logits.cpu()); ys.append(yb); idx.append(ib)
    return torch.cat(outs).numpy(), torch.cat(ys).numpy(), torch.cat(idx).numpy()

def evaluate(model, loader, criterion, dev):
    logits, y, _ = predict_logits(model, loader, dev)
    lt, yt, w = torch.from_numpy(logits), torch.from_numpy(y), criterion.weight.detach().cpu()
    loss = float(F.cross_entropy(lt, yt, weight=w, reduction='sum') / w[yt].sum())
    pred = logits.argmax(1)
    return dict(loss=loss, qwk=qwk_score(y, pred), n_pred_classes=int(len(np.unique(pred))), logits=logits, y=y)

def train_one_epoch(model, optimizer, scheduler, loader, criterion, dev, gen, scaler, st, tag):
    set_train_mode(model)
    cw = criterion.weight
    params = [p for p in model.parameters() if p.requires_grad]
    tot_loss, tot_w, n_ok = torch.zeros((), device=dev), torch.zeros((), device=dev), 0
    for xb, yb, _ in loader:
        x = normalize(gpu_augment(to_float(xb, dev), gen), model)
        y = yb.to(dev, non_blocking=True)
        wsum = cw[y].sum()
        optimizer.zero_grad(set_to_none=True)
        step_loss, bad = 0.0, False
        for xc, yc in zip(x.split(st['micro_bs']), y.split(st['micro_bs'])):
            if model._cl: xc = xc.contiguous(memory_format=torch.channels_last)
            with torch.autocast('cuda', dtype=torch.float16, enabled=CFG.amp):
                out = model(xc)
            loss = criterion(out.float(), yc) / wsum
            lv = loss.item()
            if not math.isfinite(lv):
                bad = True; break
            scaler.scale(loss).backward()
            step_loss += lv
        updated = False
        if bad:
            optimizer.zero_grad(set_to_none=True)
            st['skipped'] += 1; st['window'].append(1)
            n_fixed = repair_buffers(model, st['backup'])
            log(tag, f'non-finite loss ({lv}) -> step skipped (total skipped {st["skipped"]})' + (f'; restored {n_fixed} poisoned BatchNorm buffer(s)' if n_fixed else ''))
        else:
            tot_loss += step_loss * wsum; tot_w += wsum; n_ok += 1
            scaler.unscale_(optimizer)
            gn = torch.nn.utils.clip_grad_norm_(params, CFG.grad_clip if CFG.grad_clip else float('inf'))
            if not CFG.amp and not bool(torch.isfinite(gn)):
                optimizer.zero_grad(set_to_none=True)
                st['skipped'] += 1; st['window'].append(1)
                log(tag, f'non-finite gradient norm -> update skipped (total skipped {st["skipped"]})')
            else:
                s0 = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                updated = scaler.get_scale() >= s0
                if not updated:
                    st['scaler_skips'] += 1
                    if CFG.amp and scaler.get_scale() < 1.0:
                        raise RuntimeError(f'{tag}: GradScaler loss scale collapsed below 1 - gradients are persistently non-finite. Stopping.')
                st['window'].append(0)
        if updated:
            scheduler.step()
        w = st['window']
        if len(w) == w.maxlen and sum(w) > CFG.nf_frac * w.maxlen:
            if not st['lr_halved']:
                scheduler.scale *= 0.5; scheduler._apply(); st['lr_halved'] = True; w.clear()
                log(tag, f'>{100*CFG.nf_frac:.0f}% non-finite steps in the last {w.maxlen} -> learning rate halved (once)')
            else:
                raise RuntimeError(f'{tag}: persistent numerical instability (>{100*CFG.nf_frac:.0f}% non-finite steps after halving the LR). Stopping.')
    return float(tot_loss / tot_w) if n_ok else float('nan')

def fit(model_name, model, optimizer, scheduler, train_loader, val_loader, criterion, device, epochs, ckpt_path, meta, seed_offset=0):
    dev = torch.device(device)
    tag = f'{MODELS[model_name]["display"]}@{device}'
    gen = torch.Generator(device=dev); gen.manual_seed(CFG.seed + 1000 + seed_offset)
    scaler = make_scaler()
    st = dict(micro_bs=CFG.init_micro_batch, backup=None, ooms=0, skipped=0, scaler_skips=0, lr_halved=False, window=deque(maxlen=CFG.nf_window))
    history, best_qwk, best_epoch, ep = [], -1.0, -1, 0
    t_start = time.time()
    while ep < epochs:
        snap = take_snapshot(model, optimizer, scaler, scheduler)
        st['backup'] = snap['model']
        rng_snap = dict(aug=gen.get_state(), loader=(train_loader.generator.get_state() if getattr(train_loader, 'generator', None) is not None else None))
        guard_snap = (st['skipped'], st['scaler_skips'], st['lr_halved'])
        t0 = time.time(); torch.cuda.reset_peak_memory_stats(dev)
        oom = False
        try:
            tr_loss = train_one_epoch(model, optimizer, scheduler, train_loader, criterion, dev, gen, scaler, st, tag)
            va = evaluate(model, val_loader, criterion, dev)
        except OOM_ERRORS:
            oom = True
        if oom:
            gc.collect(); torch.cuda.empty_cache()
            optimizer.zero_grad(set_to_none=True)
            st['ooms'] += 1
            if st['ooms'] > CFG.max_oom_reductions or st['micro_bs'] <= 1:
                raise RuntimeError(f'{tag}: CUDA OOM persists after {CFG.max_oom_reductions} micro-batch reductions.')
            st['micro_bs'] //= 2
            restore_snapshot(model, optimizer, scaler, scheduler, snap)
            gen.set_state(rng_snap['aug'])
            if rng_snap['loader'] is not None: train_loader.generator.set_state(rng_snap['loader'])
            st['skipped'], st['scaler_skips'], st['lr_halved'] = guard_snap
            st['window'].clear()
            log(tag, f'CUDA OOM -> model/optimizer/scaler/scheduler + augmentation & shuffle RNG restored to epoch start; micro-batch now {st["micro_bs"]} x {CFG.batch_size // st["micro_bs"]} accumulation steps (reduction {st["ooms"]}/{CFG.max_oom_reductions})')
            continue
        del snap
        ep += 1
        dt = time.time() - t0
        rec = dict(epoch=ep, train_loss=tr_loss, val_loss=va['loss'], val_qwk=va['qwk'], lr_backbone=optimizer.param_groups[0]['lr'],
                   lr_head=optimizer.param_groups[-1]['lr'], epoch_seconds=dt, gpu_peak_gb=torch.cuda.max_memory_allocated(dev) / 2**30,
                   n_pred_classes=va['n_pred_classes'], micro_batch=st['micro_bs'], skipped_steps=st['skipped'], scaler_skipped_updates=st['scaler_skips'],
                   cpu_ram_used_gb=resource_snapshot()['cpu_ram_used_gb'])
        is_best = va['qwk'] > best_qwk
        if is_best:
            best_qwk, best_epoch = va['qwk'], ep
            torch.save(dict(state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()},
                            epoch=ep, val_qwk=va['qwk'], **meta), ckpt_path)
        rec['is_best'] = bool(is_best)
        history.append(rec)
        log(tag, f'ep {ep:02d}/{epochs} | train {tr_loss:.4f} | val {va["loss"]:.4f} | QWK {va["qwk"]:.4f} | lr {rec["lr_backbone"]:.1e}/{rec["lr_head"]:.1e} | '
                 f'{dt:.0f}s | gpu {rec["gpu_peak_gb"]:.1f}GB | classes {va["n_pred_classes"]}/5 | mb {st["micro_bs"]}{" *best*" if is_best else ""}')
        if ep == 1:
            log(tag, f'measured epoch time {dt:.0f}s -> estimated total for this model: ~{dt * epochs / 60:.0f} min for {epochs} epochs')
        if va['n_pred_classes'] < 4:
            log(tag, f'!!! WARNING: development predictions use only {va["n_pred_classes"]} of 5 classes (predictions are NOT modified)')
    return dict(history=history, best_qwk=best_qwk, best_epoch=best_epoch, train_seconds=time.time() - t_start,
                oom_reductions=st['ooms'], skipped_steps=st['skipped'], scaler_skipped_updates=st['scaler_skips'], final_micro_batch=st['micro_bs'])

RESULTS = {}
BG_THREADS = {}

def run_signature(key):
    cfg_keys = ('epochs', 'batch_size', 'warmup_epochs', 'min_lr_factor', 'lr_backbone', 'lr_head', 'weight_decay', 'grad_clip', 'amp', 'init_micro_batch',
                'aug_rotation_range_deg', 'aug_brightness', 'aug_contrast', 'aug_crop_scale', 'aug_crop_ratio_jitter', 'seed', 'img_size', 'split_fracs')
    return make_sig(key, MODELS[key]['timm_id'], timm.__version__, RULES[key], {k: getattr(CFG, k) for k in cfg_keys}, PRE_SIG, DATA_SIG, MANIFEST_HASH, EYE_SUBSET_SIZE,
                    [round(float(x), 6) for x in CLASS_W.tolist()], len(DF['train']), len(DF['val']))

def finished_result(key):
    ck, hp = os.path.join(PATHS.out, MODELS[key]['ckpt']), os.path.join(PATHS.out, f'history_{key}.json')
    if not (CFG.reuse_finished and os.path.exists(ck) and os.path.exists(hp)):
        return None
    meta = torch.load(ck, map_location='cpu', weights_only=False)
    if meta.get('run_sig') != run_signature(key):
        log(MODELS[key]['display'], 'existing checkpoint has a different run signature - NOT reusing it, retraining')
        return None
    res = json.load(open(hp))
    log(MODELS[key]['display'], f'found finished run with identical signature (best val QWK {res["best_qwk"]:.4f} @ epoch {res["best_epoch"]}) - reusing it instead of retraining')
    return res

def run_job(key, device_str, seed_offset):
    res = finished_result(key)
    if res is None:
        dev = torch.device(device_str)
        with torch.cuda.device(dev):
            model = build_model(key).to(dev)
            if model._cl: model = model.to(memory_format=torch.channels_last)
            optimizer = make_optimizer(model, key)
            tl = make_loader(DF['train'], True, seed=CFG.seed + seed_offset)
            vl = make_loader(DF['val'], False)
            spe = len(tl)
            scheduler = WarmupCosine(optimizer, CFG.warmup_epochs * spe, CFG.epochs * spe, CFG.min_lr_factor)
            criterion = nn.CrossEntropyLoss(weight=CLASS_W.to(dev), reduction='sum')
            meta = dict(model_key=key, timm_id=MODELS[key]['timm_id'], epochs=CFG.epochs, seed=CFG.seed,
                        n_train=len(DF['train']), n_val=len(DF['val']), img_size=CFG.img_size, run_sig=run_signature(key))
            log(MODELS[key]['display'], f'start | {spe} steps/epoch | trainable {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M params | {torch.cuda.get_device_name(dev)}')
            res = fit(key, model, optimizer, scheduler, tl, vl, criterion, device_str, CFG.epochs,
                      os.path.join(PATHS.out, MODELS[key]['ckpt']), meta, seed_offset)
            json.dump(res, open(os.path.join(PATHS.out, f'history_{key}.json'), 'w'), indent=1)
            del model, optimizer, tl, vl, scheduler; gc.collect(); torch.cuda.empty_cache()
    RESULTS[key] = res
    log(MODELS[key]['display'], f'DONE | best val QWK {res["best_qwk"]:.4f} @ epoch {res["best_epoch"]} | {res["train_seconds"]/60:.1f} min | OOM reductions {res["oom_reductions"]} | non-finite-loss steps {res["skipped_steps"]} | GradScaler-skipped updates {res.get("scaler_skipped_updates", 0)}')
    return res

def launch_background(key, device_str, seed_offset):
    box = dict(error=None)
    def target():
        try: run_job(key, device_str, seed_offset)
        except BaseException as e:
            box['error'] = e; log(key, f'FAILED: {e!r}')
    th = threading.Thread(target=target, name=f'train-{key}', daemon=True)
    th.start(); BG_THREADS[key] = (th, box)

def join_background(key):
    th, box = BG_THREADS[key]; th.join()
    if box['error'] is not None:
        raise RuntimeError(f'background training of {key} failed') from box['error']
print('training utilities ready | effective batch', CFG.batch_size, '| initial micro-batch', CFG.init_micro_batch)

In [ ]:
TRAIN_T0 = time.time()
if PARALLEL:
    launch_background('efficientnet_b0', 'cuda:1', seed_offset=1)
    print('EfficientNet-B0 launched in the background on cuda:1')
run_job('resnet50', 'cuda:0', seed_offset=0)
print(resource_snapshot())

In [ ]:
if 'efficientnet_b0' in BG_THREADS:
    join_background('efficientnet_b0')
else:
    run_job('efficientnet_b0', 'cuda:0', seed_offset=1)
print(resource_snapshot())

In [ ]:
run_job('vit_tiny', 'cuda:0', seed_offset=2)
TOTAL_TRAIN_WALL_SECONDS = time.time() - TRAIN_T0
TOTAL_TRAIN_SUM_SECONDS = sum(r['train_seconds'] for r in RESULTS.values())
TIMES.update({f'train_{k}_s': RESULTS[k]['train_seconds'] for k in MODEL_KEYS}); TIMES['train_wall_s'] = TOTAL_TRAIN_WALL_SECONDS
print(f'Training wall-clock: {TOTAL_TRAIN_WALL_SECONDS/60:.1f} min | sum of per-model training durations: {TOTAL_TRAIN_SUM_SECONDS/60:.1f} min')
print(resource_snapshot())
assert set(RESULTS) == set(MODEL_KEYS)

In [ ]:
COLORS = {'resnet50': '#2a78d6', 'efficientnet_b0': '#eb6834', 'vit_tiny': '#1baf7a', 'ensemble': '#4a3aa7'}
MARKERS = {'resnet50': 'o', 'efficientnet_b0': 's', 'vit_tiny': '^', 'ensemble': 'D'}
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.25,
                     'axes.titlesize': 11, 'axes.labelsize': 10, 'figure.dpi': 110, 'savefig.dpi': 160, 'savefig.bbox': 'tight'})
DISPLAY = {**{k: v['display'] for k, v in MODELS.items()}, 'ensemble': 'Ensemble'}

def plot_training_curves(path=None):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    for k in MODEL_KEYS:
        h = pd.DataFrame(RESULTS[k]['history'])
        ax[0].plot(h.epoch, h.val_qwk, color=COLORS[k], marker=MARKERS[k], ms=5, lw=2, label=DISPLAY[k])
        b = h.loc[h.val_qwk.idxmax()]
        ax[0].scatter([b.epoch], [b.val_qwk], s=140, facecolors='none', edgecolors=COLORS[k], linewidths=2)
        ax[1].plot(h.epoch, h.train_loss, color=COLORS[k], lw=2, label=f'{DISPLAY[k]} train')
        ax[1].plot(h.epoch, h.val_loss, color=COLORS[k], lw=2, ls='--', label=f'{DISPLAY[k]} val')
    ax[0].set(title='Development QWK by epoch (ring = best epoch / saved checkpoint)', xlabel='Epoch', ylabel='Quadratic weighted kappa')
    ax[1].set(title='Class-weighted cross-entropy', xlabel='Epoch', ylabel='Loss')
    ax[0].legend(frameon=False); ax[1].legend(frameon=False, fontsize=8, ncol=2)
    if path: fig.savefig(path)
    plt.show()

plot_training_curves(os.path.join(PATHS.out, 'training_curves.png'))
print(pd.DataFrame([dict(model=DISPLAY[k], best_val_qwk=round(RESULTS[k]['best_qwk'], 4), best_epoch=RESULTS[k]['best_epoch'],
                         classes_predicted_at_best=[h['n_pred_classes'] for h in RESULTS[k]['history'] if h['epoch'] == RESULTS[k]['best_epoch']][0],
                         minutes=round(RESULTS[k]['train_seconds'] / 60, 1), oom_reductions=RESULTS[k]['oom_reductions'],
                         skipped_steps=RESULTS[k]['skipped_steps']) for k in MODEL_KEYS]).to_string(index=False))
for k in MODEL_KEYS:
    low = [h['epoch'] for h in RESULTS[k]['history'] if h['n_pred_classes'] < 4]
    if low:
        print(f'!!! WARNING: {DISPLAY[k]} used fewer than 4 of 5 classes on the development set at epochs {low}')

In [ ]:
DEV0 = torch.device('cuda:0')
LOADED = OrderedDict()
for key in MODEL_KEYS:
    ck = torch.load(os.path.join(PATHS.out, MODELS[key]['ckpt']), map_location='cpu', weights_only=False)
    assert ck['model_key'] == key and ck['timm_id'] == MODELS[key]['timm_id']
    assert ck['epoch'] == RESULTS[key]['best_epoch'] and abs(ck['val_qwk'] - RESULTS[key]['best_qwk']) < 1e-9, 'checkpoint does not match training history'
    m = build_model(key, pretrained=False)
    m.load_state_dict(ck['state_dict'])
    LOADED[key] = m.eval()
    print(f'{DISPLAY[key]:16s} loaded best checkpoint: epoch {ck["epoch"]}, val QWK {ck["val_qwk"]:.4f}')

In [ ]:
def softmax_np(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def predict_split(frame, name, label_col='label'):
    loader = make_loader(frame, False)
    out = dict(y=frame[label_col].values.astype(np.int64), patient=frame.patient_id.values.astype(str),
               image=frame.image_id.values.astype(str), logits={}, probs={})
    for key in MODEL_KEYS:
        LOADED[key].to(DEV0)
        try:
            logits, y, idx = predict_logits(LOADED[key], loader, DEV0)
        finally:
            LOADED[key].cpu(); torch.cuda.empty_cache()
        assert (idx == np.arange(len(frame))).all() and (y == out['y']).all() and np.isfinite(logits).all() and logits.shape == (len(frame), CFG.num_classes)
        out['logits'][key] = logits; out['probs'][key] = softmax_np(logits)
        np.savez_compressed(os.path.join(PATHS.preds, f'{name}_{key}.npz'), logits=logits, probabilities=out['probs'][key],
                            labels=out['y'], patient_ids=out['patient'], image_ids=out['image'])
    return out

PRED = {}
_t = time.time()
PRED['val'] = predict_split(DF['val'], 'val')
TIMES['val_inference_s'] = time.time() - _t
for k in MODEL_KEYS:
    print(f'{DISPLAY[k]:16s} val argmax-QWK {qwk_score(PRED["val"]["y"], PRED["val"]["logits"][k].argmax(1)):.4f}')

In [ ]:
_t_ens0 = time.time()
K = CFG.num_classes

def nll_score(y, P):
    return float(-np.mean(np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1.0))))

def brier_score(y, P):
    return float(np.mean(((P - np.eye(K)[y]) ** 2).sum(1)))

def reliability_bins(y, P, bins=CFG.ece_bins):
    conf, acc = P.max(1), (P.argmax(1) == y).astype(np.float64)
    b = np.minimum((conf * bins).astype(int), bins - 1)
    cnt = np.bincount(b, minlength=bins).astype(float)
    sc, sa = np.bincount(b, weights=conf, minlength=bins), np.bincount(b, weights=acc, minlength=bins)
    with np.errstate(invalid='ignore', divide='ignore'):
        return cnt, sc / cnt, sa / cnt            # per-bin count, mean confidence, accuracy (NaN for empty bins)

def ece_score(y, P, bins=CFG.ece_bins):
    conf, acc = P.max(1), (P.argmax(1) == y).astype(np.float64)
    b = np.minimum((conf * bins).astype(int), bins - 1)
    return float(np.abs(np.bincount(b, weights=acc, minlength=bins) - np.bincount(b, weights=conf, minlength=bins)).sum() / len(y))

def qwk_fast(y, pred):
    cm = np.bincount(y * K + pred, minlength=K * K).reshape(K, K).astype(np.float64)
    n = cm.sum(); i, j = np.indices((K, K)); W = (i - j) ** 2 / (K - 1) ** 2
    exp = np.outer(cm.sum(1), cm.sum(0)) / n
    den = (W * exp).sum()
    return 0.0 if den == 0 else float(1 - (W * cm).sum() / den)

def referable_metrics(y, P):
    s = P[:, CFG.referable_min_grade:].sum(1)           # P(referable) = P(2) + P(3) + P(4)
    t = y >= CFG.referable_min_grade
    pred = s >= CFG.referable_threshold
    sens = float((pred & t).sum() / max(1, t.sum())); spec = float((~pred & ~t).sum() / max(1, (~t).sum()))
    auc = float(roc_auc_score(t, s)) if 0 < t.sum() < len(t) else float('nan')
    return dict(ref_sensitivity=sens, ref_specificity=spec, ref_auc=auc)

def scalar_metrics(y, P):
    return dict(qwk=qwk_fast(y, P.argmax(1)), **referable_metrics(y, P), ece=ece_score(y, P), nll=nll_score(y, P), brier=brier_score(y, P))

def full_metrics(y, P):
    pred = P.argmax(1)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=list(range(K)), zero_division=0)
    return dict(**scalar_metrics(y, P), accuracy=float((pred == y).mean()), precision=pr.tolist(), recall=rc.tolist(), f1=f1.tolist(),
                support=sup.tolist(), confusion=confusion_matrix(y, pred, labels=list(range(K))).tolist(), n=int(len(y)))

# sanity-check the fast QWK against scikit-learn on the real development predictions
for _k in MODEL_KEYS:
    _p = PRED['val']['logits'][_k].argmax(1)
    assert abs(qwk_fast(PRED['val']['y'], _p) - cohen_kappa_score(PRED['val']['y'], _p, weights='quadratic')) < 1e-9

def fit_temperature(logits, y):
    lt, yt = torch.tensor(logits, dtype=torch.float64), torch.tensor(y)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=300, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(lt / log_t.exp(), yt); loss.backward(); return loss
    opt.step(closure)
    return float(log_t.exp())

T_MIN, T_MAX = 0.05, 20.0

def fit_temperature_safe(lg, y, name='', verbose=True):
    T = fit_temperature(lg, y)
    before, after = nll_score(y, softmax_np(lg)), nll_score(y, softmax_np(lg, T))
    if not (T_MIN < T < T_MAX) or after > before + 1e-9:
        if verbose: print(f'WARNING: temperature fit for {name} unreliable (T={T:.3f}, NLL {before:.4f}->{after:.4f}); falling back to T=1')
        return 1.0
    return T

TEMPS = {}
for k in MODEL_KEYS:
    lg, y = PRED['val']['logits'][k], PRED['val']['y']
    T = fit_temperature_safe(lg, y, DISPLAY[k])
    TEMPS[k] = T
    print(f'{DISPLAY[k]:16s} T = {T:.4f} | val NLL {nll_score(y, softmax_np(lg)):.4f} -> {nll_score(y, softmax_np(lg, T)):.4f} | val ECE {ece_score(y, softmax_np(lg)):.4f} -> {ece_score(y, softmax_np(lg, T)):.4f}  (in-sample, so optimistic)')
json.dump(TEMPS, open(os.path.join(PATHS.out, 'temperatures.json'), 'w'), indent=1)

In [ ]:
def calibrate(logits_by_model):
    return {k: softmax_np(logits_by_model[k], TEMPS[k]) for k in MODEL_KEYS}

PCAL = {'val': calibrate(PRED['val']['logits'])}
for k in MODEL_KEYS:
    assert np.allclose(PCAL['val'][k].sum(1), 1) and (PCAL['val'][k].argmax(1) == PRED['val']['logits'][k].argmax(1)).all()
print('calibrated validation probabilities ready:', {k: v.shape for k, v in PCAL['val'].items()})

In [ ]:
def stack_features(P_by_model):
    return np.concatenate([P_by_model[k] for k in MODEL_KEYS], axis=1)

def lr_predict(clf, X):
    P = np.full((len(X), K), 1e-12)
    P[:, clf.classes_] = clf.predict_proba(X)
    return P / P.sum(1, keepdims=True)

yv, gv = PRED['val']['y'], PRED['val']['patient']
Xv = stack_features(PCAL['val'])
oof_mean, oof = np.zeros((len(yv), K)), np.zeros((len(yv), K))
fold_nll = []
sgkf = StratifiedGroupKFold(n_splits=CFG.cv_folds, shuffle=True, random_state=CFG.seed)
for tr_i, te_i in sgkf.split(Xv, yv, groups=gv):
    Tf = {k: fit_temperature_safe(PRED['val']['logits'][k][tr_i], yv[tr_i], verbose=False) for k in MODEL_KEYS}
    Ptr = {k: softmax_np(PRED['val']['logits'][k][tr_i], Tf[k]) for k in MODEL_KEYS}
    Pte = {k: softmax_np(PRED['val']['logits'][k][te_i], Tf[k]) for k in MODEL_KEYS}
    oof_mean[te_i] = np.mean([Pte[k] for k in MODEL_KEYS], axis=0)
    clf = LogisticRegression(C=CFG.lr_combiner_C, max_iter=5000)
    clf.fit(stack_features(Ptr), yv[tr_i])
    oof[te_i] = lr_predict(clf, stack_features(Pte))
    fold_nll.append((nll_score(yv[te_i], oof_mean[te_i]), nll_score(yv[te_i], oof[te_i])))
nll_mean = nll_score(yv, oof_mean)
nll_lr = nll_score(yv, oof)
print('per-fold held-out NLL (mean | logreg):', ' ; '.join(f'f{i + 1}: {a:.4f} | {b:.4f}' for i, (a, b) in enumerate(fold_nll)))

SELECTED_COMBINER = 'mean' if nll_mean <= nll_lr else 'logreg'
print(f'Mean probability NLL ({CFG.cv_folds}-fold grouped CV, fold-specific temperatures, validation) : {nll_mean:.4f}')
print(f'Logistic-regression combiner NLL (CV)               : {nll_lr:.4f}')
print(f'Selected combiner (lowest grouped-CV NLL)            : {SELECTED_COMBINER}   [selection objective = NLL, i.e. calibration quality; QWK is reported for information only]')
print(f'(info only) validation QWK  mean: {qwk_fast(yv, oof_mean.argmax(1)):.4f} | logreg(OOF): {qwk_fast(yv, oof.argmax(1)):.4f}')
COMBINER_CLF = None
if SELECTED_COMBINER == 'logreg':
    COMBINER_CLF = LogisticRegression(C=CFG.lr_combiner_C, max_iter=5000).fit(Xv, yv)

In [ ]:
def combine(P_by_model):
    if SELECTED_COMBINER == 'mean':
        return np.mean([P_by_model[k] for k in MODEL_KEYS], axis=0)
    return lr_predict(COMBINER_CLF, stack_features(P_by_model))

def final_pipeline(logits_by_model):
    Pc = calibrate(logits_by_model)
    return combine(Pc), Pc

_ens_val, _ = final_pipeline(PRED['val']['logits'])
assert np.allclose(_ens_val.sum(1), 1)
combiner_json = dict(selected=SELECTED_COMBINER, cv_nll_mean=nll_mean, cv_nll_logreg=nll_lr, model_order=MODEL_KEYS, cv_folds=CFG.cv_folds, C=CFG.lr_combiner_C)
if COMBINER_CLF is not None:
    combiner_json.update(classes=COMBINER_CLF.classes_.tolist(), coef=COMBINER_CLF.coef_.tolist(), intercept=COMBINER_CLF.intercept_.tolist(),
                         feature_order='[P_' + ', P_'.join(MODEL_KEYS) + '] each with 5 class columns (temperature-scaled probabilities)')
json.dump(combiner_json, open(os.path.join(PATHS.out, 'combiner.json'), 'w'), indent=1)
FINAL_CONFIG = dict(temperatures=TEMPS, combiner=SELECTED_COMBINER)
print('FINAL SYSTEM:', json.dumps(FINAL_CONFIG))
print('saved temperatures.json and combiner.json')
TIMES['calibration_ensemble_s'] = time.time() - _t_ens0

In [ ]:
ENS = {}
def apply_final(name, pred):
    ENS[name], PCAL[name] = final_pipeline(pred['logits'])
    assert np.allclose(ENS[name].sum(1), 1) and np.isfinite(ENS[name]).all()

_t = time.time()
PRED['test'] = predict_split(DF['test'], 'test')
TIMES['eyepacs_test_inference_s'] = time.time() - _t
apply_final('test', PRED['test'])
print('test images:', len(PRED['test']['y']), '| test patients:', len(set(PRED['test']['patient'])))

In [ ]:
# ===== CELL 35 — EYEPACS METRICS (QWK, per-class precision/recall/F1, referable, calibration) =====
METRICS = {}
SYSTEMS = MODEL_KEYS + ['ensemble']

def compute_metrics(name, pred):
    y = pred['y']
    ts = {k: full_metrics(y, PCAL[name][k]) for k in MODEL_KEYS}; ts['ensemble'] = full_metrics(y, ENS[name])
    raw = {k: full_metrics(y, pred['probs'][k]) for k in MODEL_KEYS}
    METRICS[name] = dict(ts=ts, raw=raw)

def results_table(name):
    rows = []
    for s in SYSTEMS:
        m = METRICS[name]['ts'][s]
        rows.append({'Model': DISPLAY[s], 'QWK': m['qwk'], **{f'Grade-{c} Recall': m['recall'][c] for c in range(K)},
                     'Ref Sensitivity': m['ref_sensitivity'], 'Ref Specificity': m['ref_specificity'], 'AUC': m['ref_auc'],
                     'ECE': m['ece'], 'NLL': m['nll'], 'Brier': m['brier']})
    return pd.DataFrame(rows).set_index('Model')

def per_class_table(name, s):
    m = METRICS[name]['ts'][s]
    return pd.DataFrame({'Precision': m['precision'], 'Recall': m['recall'], 'F1': m['f1'], 'Support': m['support']},
                        index=[f'Grade {c} ({CFG.class_names[c]})' for c in range(K)])

compute_metrics('test', PRED['test'])
print('=== EyePACS TEST — quadratic weighted kappa (primary 5-class metric) ===')
print(results_table('test')[['QWK']].round(4).to_string())
for s in SYSTEMS:
    print(f'\n--- per-class metrics: {DISPLAY[s]} ---')
    print(per_class_table('test', s).round(4).to_string())

In [ ]:
# ===== CELL 36 — CONFUSION MATRIX (ensemble) =====
def plot_confusion(cm, title, path=None):
    cm = np.asarray(cm); rown = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(6.2, 5.4))
    im = ax.imshow(rown, cmap='Blues', vmin=0, vmax=1)
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f'{cm[i, j]}\n{100*rown[i, j]:.0f}%', ha='center', va='center', fontsize=9, color='white' if rown[i, j] > 0.55 else '#0b0b0b')
    ax.set(xticks=range(K), yticks=range(K), xlabel='Predicted grade', ylabel='Actual grade', title=title); ax.grid(False)
    fig.colorbar(im, ax=ax, label='Row-normalised share')
    if path: fig.savefig(path)
    plt.show()

plot_confusion(METRICS['test']['ts']['ensemble']['confusion'], 'EyePACS test — ensemble confusion matrix (count, % of actual)', os.path.join(PATHS.out, 'confusion_matrix.png'))
for s in MODEL_KEYS:
    print(f'\n{DISPLAY[s]} confusion matrix (rows=actual, cols=predicted)')
    print(pd.DataFrame(METRICS['test']['ts'][s]['confusion'], index=[f'Actual {c}' for c in range(K)], columns=[f'Pred {c}' for c in range(K)]).to_string())

In [ ]:
# ===== CELL 37 — CALIBRATION EVALUATION (15-bin ECE, NLL, Brier, reliability diagram) =====
def calibration_table(name):
    rows = []
    for k in MODEL_KEYS:
        for label, key in (('raw', 'raw'), ('temperature-scaled', 'ts')):
            m = METRICS[name][key][k]
            rows.append(dict(System=DISPLAY[k], Variant=label, ECE=m['ece'], NLL=m['nll'], Brier=m['brier']))
    m = METRICS[name]['ts']['ensemble']
    rows.append(dict(System='Ensemble', Variant='TS + combiner', ECE=m['ece'], NLL=m['nll'], Brier=m['brier']))
    return pd.DataFrame(rows)

def plot_reliability(name, pred, path=None):
    y = pred['y']; fig, axes = plt.subplots(1, 4, figsize=(17, 4.4), sharex=True, sharey=True)
    for ax, s in zip(axes, SYSTEMS):
        ax.plot([0, 1], [0, 1], color='#8a8a85', lw=1, ls=':', label='perfect')
        if s != 'ensemble':
            _, c, a = reliability_bins(y, pred['probs'][s]); ok = ~np.isnan(c)
            ax.plot(c[ok], a[ok], color=COLORS[s], lw=1.5, ls='--', marker=MARKERS[s], mfc='none', ms=6, label=f'raw (ECE {METRICS[name]["raw"][s]["ece"]:.3f})')
            P = PCAL[name][s]
        else:
            P = ENS[name]
        _, c, a = reliability_bins(y, P); ok = ~np.isnan(c)
        lab = f'temp-scaled (ECE {METRICS[name]["ts"][s]["ece"]:.3f})' if s != 'ensemble' else f'ensemble (ECE {METRICS[name]["ts"][s]["ece"]:.3f})'
        ax.plot(c[ok], a[ok], color=COLORS[s], lw=2, marker=MARKERS[s], ms=6, label=lab)
        ax.set(title=DISPLAY[s], xlabel='Mean confidence (max prob., 15 bins)', xlim=(0, 1), ylim=(0, 1)); ax.legend(frameon=False, fontsize=8, loc='upper left')
    axes[0].set_ylabel('Accuracy')
    fig.suptitle(f'Reliability diagrams — {name}', y=1.02)
    if path: fig.savefig(path)
    plt.show()

print(calibration_table('test').round(4).to_string(index=False))
plot_reliability('test', PRED['test'], os.path.join(PATHS.out, 'calibration.png'))
_ens = METRICS['test']['ts']['ensemble']
for _k in MODEL_KEYS:
    r, t = METRICS['test']['raw'][_k], METRICS['test']['ts'][_k]
    print(f'{DISPLAY[_k]:16s} temperature scaling on test: ECE {r["ece"]:.4f} -> {t["ece"]:.4f} ({"improved" if t["ece"] < r["ece"] else "NOT improved"}), NLL {r["nll"]:.4f} -> {t["nll"]:.4f} ({"improved" if t["nll"] < r["nll"] else "NOT improved"})')
_best_ts_ece = min(METRICS['test']['ts'][k]['ece'] for k in MODEL_KEYS)
print(f'Ensemble ECE {_ens["ece"]:.4f} vs best temperature-scaled single-model ECE {_best_ts_ece:.4f} -> {"lower" if _ens["ece"] < _best_ts_ece else "NOT lower"} (point estimates; see CIs in Cell 39)')

In [ ]:
# ===== CELL 38 — REFERABLE DR EVALUATION (grade >= 2; score = P(2)+P(3)+P(4); decision threshold fixed at 0.5 a priori) =====
def referable_table(name):
    rows = []
    for s in SYSTEMS:
        m = METRICS[name]['ts'][s]
        rows.append({'Model': DISPLAY[s], 'Sensitivity': m['ref_sensitivity'], 'Specificity': m['ref_specificity'], 'AUC': m['ref_auc']})
    return pd.DataFrame(rows).set_index('Model')

_yt = PRED['test']['y']
print(f'EyePACS test: referable prevalence = {100*(_yt >= 2).mean():.2f}% ({int((_yt >= 2).sum())}/{len(_yt)})')
print(referable_table('test').round(4).to_string())

In [ ]:
# ===== CELL 39 — PATIENT-LEVEL BOOTSTRAP 95% CONFIDENCE INTERVALS =====
BOOT_METRICS = ['qwk', 'ref_sensitivity', 'ref_specificity', 'ref_auc', 'ece', 'nll', 'brier']
BOOT_NAMES = dict(qwk='QWK', ref_sensitivity='Referable sensitivity', ref_specificity='Referable specificity', ref_auc='Referable AUC', ece='ECE (15-bin, top-label)', nll='NLL', brier='Brier')
LOWER_IS_BETTER = {'ece', 'nll', 'brier'}
BEST_SINGLE = max(MODEL_KEYS, key=lambda k: RESULTS[k]['best_qwk'])        # chosen on DEVELOPMENT QWK, never on test results

def patient_bootstrap(y, systems, group_ids, B, seed):
    """Resample whole patients (all of their eyes) with replacement; the same resamples are used for every system (paired)."""
    uniq, inv = np.unique(group_ids, return_inverse=True)
    order = np.argsort(inv, kind='stable'); bounds = np.concatenate([[0], np.cumsum(np.bincount(inv))])
    groups = [order[bounds[i]:bounds[i + 1]] for i in range(len(uniq))]
    rng = np.random.default_rng(seed)
    out = {s: {m: np.full(B, np.nan) for m in BOOT_METRICS} for s in systems}
    for b in range(B):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]); yb = y[idx]
        for s, P in systems.items():
            sm = scalar_metrics(yb, P[idx])
            for m in BOOT_METRICS: out[s][m][b] = sm[m]
    return out

def describe_delta(metric, point, draws):
    lo, hi = np.nanpercentile(draws, [2.5, 97.5])
    good = (point < 0 and hi < 0) if metric in LOWER_IS_BETTER else (point > 0 and lo > 0)
    bad = (point > 0 and lo > 0) if metric in LOWER_IS_BETTER else (point < 0 and hi < 0)
    verdict = 'paired 95% CI excludes 0 (favours the ensemble)' if good else ('paired 95% CI excludes 0 (favours the best single model)' if bad else 'paired 95% CI includes 0 (no clear difference)')
    return dict(metric=BOOT_NAMES[metric], delta=point, lo=lo, hi=hi, verdict=verdict)

def run_bootstrap(name, pred, group_ids, unit):
    sysP = {**{k: PCAL[name][k] for k in MODEL_KEYS}, 'ensemble': ENS[name]}
    t0 = time.time(); draws = patient_bootstrap(pred['y'], sysP, group_ids, CFG.boot_reps, CFG.boot_seed)
    rows = []
    for s in SYSTEMS:
        for m in BOOT_METRICS:
            lo, hi = np.nanpercentile(draws[s][m], [2.5, 97.5])
            rows.append(dict(system=DISPLAY[s], metric=BOOT_NAMES[m], point=METRICS[name]['ts'][s][m], ci_low=lo, ci_high=hi))
    deltas = [describe_delta(m, METRICS[name]['ts']['ensemble'][m] - METRICS[name]['ts'][BEST_SINGLE][m], draws['ensemble'][m] - draws[BEST_SINGLE][m]) for m in BOOT_METRICS]
    BOOT[name] = dict(ci=pd.DataFrame(rows), delta=pd.DataFrame(deltas), unit=unit, reps=CFG.boot_reps)
    print(f'{name}: {CFG.boot_reps} bootstrap resamples over {len(np.unique(group_ids))} {unit} units ({time.time()-t0:.0f}s, seed {CFG.boot_seed})')

BOOT = {}
run_bootstrap('test', PRED['test'], PRED['test']['patient'], 'patient')
_ci = BOOT['test']['ci'].copy()
_ci['95% CI'] = _ci.apply(lambda r: f'[{r.ci_low:.4f}, {r.ci_high:.4f}]', axis=1)
print(_ci[['system', 'metric', 'point', '95% CI']].round(4).to_string(index=False))
print(f'\nPaired patient-bootstrap: Ensemble minus best single model by development QWK ({DISPLAY[BEST_SINGLE]})')
print(BOOT['test']['delta'].round(4).to_string(index=False))

In [ ]:
ap = pd.read_csv(APTOS_CSV)
print('APTOS labels file:', APTOS_CSV, '| rows', len(ap), '| columns', list(ap.columns))
ap['id_code'] = ap['id_code'].astype(str)
assert ap.id_code.is_unique, 'duplicate APTOS ids'

def locate_aptos_images(ids, csv_path):
    best = ({}, None)
    for root in (os.path.dirname(csv_path), os.path.dirname(os.path.dirname(csv_path))):
        if os.path.normpath(root) in (os.path.normpath(PATHS.input), os.path.normpath(PATHS.data_root)) or not root.startswith((PATHS.input, PATHS.data_root)):
            continue
        found = {}
        for dp, dn, fn in os.walk(root, followlinks=True):
            for f in fn:
                stem, ext = os.path.splitext(f)
                if ext.lower() in IMG_EXT and stem in ids and (stem not in found or 'train' in dp.lower()):
                    found[stem] = os.path.join(dp, f)
        if len(found) > len(best[0]): best = (found, root)
        if len(found) >= 0.99 * len(ids): break
    return best

_direct = {}
for _root in (os.path.dirname(APTOS_CSV), os.path.dirname(os.path.dirname(APTOS_CSV))):
    _td = os.path.join(_root, 'train_images')
    if os.path.isdir(_td):
        _names = {os.path.splitext(f)[0]: f for f in os.listdir(_td) if f.lower().endswith(IMG_EXT)}
        _direct = {i: os.path.join(_td, _names[i]) for i in set(ap.id_code) if i in _names}
        if len(_direct) >= 0.99 * len(ap): break
if len(_direct) >= 0.99 * len(ap):
    _found, APTOS_ROOT = _direct, os.path.dirname(_td)
else:
    _found, APTOS_ROOT = locate_aptos_images(set(ap.id_code), APTOS_CSV)
print('APTOS image root:', APTOS_ROOT)
AP_SIG = make_sig(file_md5(APTOS_CSV), sorted((i, os.path.basename(pth), os.path.getsize(pth)) for i, pth in _found.items()))
PATHS.cache_aptos = os.path.join(PATHS.cache_root, f'aptos_{CFG.img_size}_{PRE_SIG}_{AP_SIG}')
os.makedirs(PATHS.cache_aptos, exist_ok=True)
ap['src_path'] = ap.id_code.map(_found).fillna('')
_missing = int((ap.src_path == '').sum())
print('APTOS images found:', len(ap) - _missing, '/', len(ap), '| image dir(s):', sorted({os.path.dirname(p) for p in ap.src_path[ap.src_path != ''].head(50)})[:3])
if _missing / len(ap) > CFG.max_missing_frac:
    raise RuntimeError(f'{_missing} APTOS labelled images were not found in the APTOS dataset folder - the APTOS copy is incomplete (delete it and re-run to download it again).')
ap = ap[ap.src_path != ''].reset_index(drop=True)
ap['label'] = ap['diagnosis'].astype(int)
assert set(ap.label.unique()) <= set(range(K)), ap.label.unique()
print('APTOS label semantics: diagnosis 0=No DR, 1=Mild, 2=Moderate, 3=Severe, 4=Proliferative (same 5-grade scale as EyePACS)')
print(ap.label.value_counts().sort_index().rename('n').to_frame().assign(pct=lambda d: (100 * d.n / d.n.sum()).round(2)))

In [ ]:
ap['patient_id'] = ap['id_code']          # APTOS provides no patient identifier -> each image is its own bootstrap unit (see limitations)
ap['image_id'] = ap['id_code']
ap['image_path'] = PATHS.cache_aptos + '/' + ap.image_id + '.jpg'
_items = [(r.image_id, -1, r.src_path, r.image_path) for r in ap.itertuples()]
_t = time.time()
_status, _fails = build_cache(_items, 'aptos')
TIMES['preprocess_aptos_s'] = time.time() - _t
cache_report('aptos', _status, _fails, PATHS.cache_aptos, len(_items))
ap['cached'] = [os.path.exists(p) and os.path.getsize(p) > 0 for p in ap.image_path]
if int((~ap.cached).sum()) / len(ap) > CFG.max_fail_frac:
    raise RuntimeError('too many APTOS images failed preprocessing')
if (~ap.cached).any():
    print('WARNING: APTOS images excluded after failed preprocessing:', ap.loc[~ap.cached, 'image_id'].tolist()[:20])
ap = ap[ap.cached].reset_index(drop=True)
DF['aptos'] = ap
assert set(ap.image_id).isdisjoint(set(df.image_id)) and PATHS.cache_aptos != PATHS.cache_eyepacs, 'APTOS must stay completely separate from the EyePACS data'
print('APTOS images ready for evaluation:', len(ap))

In [ ]:
_t = time.time()
PRED['aptos'] = predict_split(DF['aptos'], 'aptos')
TIMES['aptos_inference_s'] = time.time() - _t
print('APTOS logits:', {k: v.shape for k, v in PRED['aptos']['logits'].items()})

In [ ]:
apply_final('aptos', PRED['aptos'])
print('applied temperatures', {k: round(v, 4) for k, v in TEMPS.items()}, '| combiner:', SELECTED_COMBINER)

In [ ]:
# ===== CELL 44 — APTOS METRICS =====
compute_metrics('aptos', PRED['aptos'])
print('=== APTOS 2019 (external) ===')
print(results_table('aptos').round(4).to_string())
print(calibration_table('aptos').round(4).to_string(index=False))
for s in ('ensemble',):
    print(f'\n--- per-class metrics: {DISPLAY[s]} ---'); print(per_class_table('aptos', s).round(4).to_string())
plot_confusion(METRICS['aptos']['ts']['ensemble']['confusion'], 'APTOS 2019 — ensemble confusion matrix (count, % of actual)', os.path.join(PATHS.out, 'confusion_matrix_aptos.png'))
run_bootstrap('aptos', PRED['aptos'], PRED['aptos']['patient'], 'image (no patient ids available)')
print('NOTE: the APTOS confidence intervals below are IMAGE-level bootstrap CIs (no patient identifiers exist), not patient-level uncertainty.')
_ci = BOOT['aptos']['ci'].copy(); _ci['95% CI'] = _ci.apply(lambda r: f'[{r.ci_low:.4f}, {r.ci_high:.4f}]', axis=1)
print(_ci[['system', 'metric', 'point', '95% CI']].round(4).to_string(index=False))

In [ ]:
# ===== CELL 45 — EYEPACS vs APTOS COMPARISON =====
CMP_METRICS = [('qwk', 'QWK'), ('ref_sensitivity', 'Ref. sensitivity'), ('ref_specificity', 'Ref. specificity'), ('ref_auc', 'Ref. AUC'),
               ('ece', 'ECE'), ('nll', 'NLL'), ('brier', 'Brier')]
_rows = []
for s in SYSTEMS:
    for m, nm in CMP_METRICS:
        e, a = METRICS['test']['ts'][s][m], METRICS['aptos']['ts'][s][m]
        _rows.append(dict(Model=DISPLAY[s], Metric=nm, EyePACS=e, APTOS=a, Change=a - e))
COMPARISON = pd.DataFrame(_rows)
for _, nm in CMP_METRICS:
    print(f'\n{nm}'); print(COMPARISON[COMPARISON.Metric == nm].drop(columns='Metric').set_index('Model').round(4).to_string())
print('\nInterpretation guide: Change = APTOS - EyePACS. Lower QWK/AUC/sensitivity or higher ECE/NLL/Brier on APTOS are reported as measured; they are consistent with')
print('camera / population / grading-protocol shift between datasets and are NOT, by themselves, evidence that the models fail or that the ensemble is robust.')

In [ ]:
# ===== CELL 46 — FINAL MODEL COMPARISON =====
FINAL_TABLE_TEST = results_table('test')
FINAL_TABLE_APTOS = results_table('aptos')
print('EyePACS test (temperature-scaled individual models; ensemble = TS + selected combiner)'); print(FINAL_TABLE_TEST.round(4).to_string())
print('\nAPTOS 2019 external'); print(FINAL_TABLE_APTOS.round(4).to_string())

CLAIMS = []
for name in ('test', 'aptos'):
    unit = BOOT[name]['unit']
    for _, r in BOOT[name]['delta'].iterrows():
        if r.metric in ('QWK', 'ECE (15-bin, top-label)', 'NLL', 'Referable AUC'):
            CLAIMS.append(f'{"EyePACS test" if name == "test" else "APTOS"} | {r.metric}: ensemble - {DISPLAY[BEST_SINGLE]} = {r.delta:+.4f} (95% CI {r.lo:+.4f} to {r.hi:+.4f}, {unit} bootstrap) -> {r.verdict}')
print('\nMeasured ensemble-vs-best-single-model statements (best single chosen on development QWK):')
for c in CLAIMS: print(' -', c)

In [ ]:
def plot_class_distribution(path=None):
    sets = {'EyePACS full': df_full.label, 'subset train': DF['train'].label, 'subset val': DF['val'].label, 'subset test': DF['test'].label, 'APTOS': DF['aptos'].label}
    cols = ['#8a8a85', '#2a78d6', '#eb6834', '#1baf7a', '#4a3aa7']
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    w = 0.16
    for i, ((nm, lab), c) in enumerate(zip(sets.items(), cols)):
        share = 100 * lab.value_counts(normalize=True).reindex(range(K)).fillna(0).values
        ax[0].bar(np.arange(K) + (i - 2) * w, share, w, color=c, label=f'{nm} (n={len(lab):,})')
    ax[0].set(title='Grade distribution (% of images)', xticks=range(K), xlabel='DR grade', ylabel='% of split'); ax[0].legend(frameon=False, fontsize=8)
    ax[0].set_xticklabels([f'{c}\n{n}' for c, n in enumerate(CFG.class_names)])
    tot = pd.concat([DF['train'].label, DF['val'].label, DF['test'].label]).value_counts().reindex(range(K))
    ax[1].bar(range(K), tot.values, color='#2a78d6'); ax[1].set_yscale('log')
    ax[1].set_ylim(1, tot.values.max() * 4)
    for i, v in enumerate(tot.values): ax[1].text(i, v * 1.15, f'{v:,}', ha='center', fontsize=9)
    ax[1].set(title='EyePACS subset used in this run: images per grade (log scale)', xticks=range(K), xlabel='DR grade', ylabel='images')
    if path: fig.savefig(path)
    plt.show()

def plot_per_class(name, path=None):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True); w = 0.2
    for a, (key, nm) in zip(ax, (('recall', 'Recall'), ('f1', 'F1'))):
        for i, s in enumerate(SYSTEMS):
            a.bar(np.arange(K) + (i - 1.5) * w, METRICS[name]['ts'][s][key], w, color=COLORS[s], label=DISPLAY[s])
        a.set(title=f'Per-class {nm} — {name}', xticks=range(K), xlabel='DR grade', ylim=(0, 1))
    ax[0].set_ylabel('score'); ax[0].legend(frameon=False, fontsize=8, ncol=2)
    if path: fig.savefig(path)
    plt.show()

def plot_comparison(path=None):
    fig, axes = plt.subplots(2, 4, figsize=(17, 7.5)); axes = axes.ravel()
    for ax, (m, nm) in zip(axes, CMP_METRICS):
        x = np.arange(len(SYSTEMS)); w = 0.38
        e = [METRICS['test']['ts'][s][m] for s in SYSTEMS]; a = [METRICS['aptos']['ts'][s][m] for s in SYSTEMS]
        ax.bar(x - w / 2, e, w, color='#2a78d6', label='EyePACS test'); ax.bar(x + w / 2, a, w, color='#eb6834', label='APTOS')
        ax.set(title=nm + (' (lower is better)' if m in LOWER_IS_BETTER else ''), xticks=x); ax.set_xticklabels([DISPLAY[s].replace('EfficientNet-B0', 'EffNet-B0') for s in SYSTEMS], fontsize=8, rotation=15)
    axes[0].legend(frameon=False, fontsize=8); axes[-1].axis('off')
    if path: fig.savefig(path)
    plt.show()

plot_class_distribution(os.path.join(PATHS.out, 'class_distribution.png'))
plot_training_curves(os.path.join(PATHS.out, 'training_curves.png'))
plot_confusion(METRICS['test']['ts']['ensemble']['confusion'], 'EyePACS test — ensemble confusion matrix (count, % of actual)', os.path.join(PATHS.out, 'confusion_matrix.png'))
plot_per_class('test', os.path.join(PATHS.out, 'per_class_performance.png'))
plot_reliability('test', PRED['test'], os.path.join(PATHS.out, 'calibration.png'))
plot_comparison(os.path.join(PATHS.out, 'eyepacs_vs_aptos.png'))

In [ ]:
def to_jsonable(o):
    if isinstance(o, dict): return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, pd.DataFrame): return o.to_dict('records')
    return o

PCAL['val'] = calibrate(PRED['val']['logits']); ENS['val'] = combine(PCAL['val'])
def pred_frame(ds, tag):
    p = PRED[ds]; f = pd.DataFrame(dict(dataset=tag, image_id=p['image'], true_grade=p['y']))
    f['ensemble_grade'] = ENS[ds].argmax(1)
    for c in range(K): f[f'ensemble_p{c}'] = ENS[ds][:, c].round(5)
    f['p_referable'] = ENS[ds][:, CFG.referable_min_grade:].sum(1).round(5)
    for k in MODEL_KEYS:
        f[f'{k}_grade'] = PCAL[ds][k].argmax(1)
        for c in range(K): f[f'{k}_p{c}'] = PCAL[ds][k][:, c].round(5)
    return f
pd.concat([pred_frame('test', 'eyepacs_test'), pred_frame('aptos', 'aptos2019_external')], ignore_index=True).to_csv(os.path.join(PATHS.out, 'predictions.csv'), index=False)
pred_frame('val', 'eyepacs_val_DEVELOPMENT_IN_SAMPLE').to_csv(os.path.join(PATHS.out, 'development_predictions_in_sample.csv'), index=False)
DEV_METRICS = dict(NOTE='in-sample: checkpoints, temperatures and the combiner were selected/fitted on this split; do not report as performance',
                   ts={k: full_metrics(PRED['val']['y'], PCAL['val'][k]) for k in MODEL_KEYS}, ensemble=full_metrics(PRED['val']['y'], ENS['val']))

json.dump(to_jsonable(RESULTS), open(os.path.join(PATHS.out, 'training_histories.json'), 'w'), indent=1)
import sklearn
try:
    _driver = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True, timeout=10).strip().split('\n')[0]
except Exception:
    _driver = 'unknown'
ENV = dict(seed=CFG.seed, python=sys.version.split()[0], torch=torch.__version__, cuda=torch.version.cuda, gpu_driver=_driver, timm=timm.__version__,
           numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, opencv=cv2.__version__, matplotlib=matplotlib.__version__,
           gpus=[torch.cuda.get_device_name(i) for i in range(NUM_GPUS)], model_ids={k: v['timm_id'] for k, v in MODELS.items()},
           eyepacs=dict(labels_md5=LABELS_MD5, archives=ARCHIVE_FINGERPRINT, cache_signature=DATA_SIG), aptos=dict(labels_md5=file_md5(APTOS_CSV), cache_signature=AP_SIG),
           preprocessing_signature=PRE_SIG, data_sources=dict(eyepacs=EYEPACS_SOURCE, aptos=APTOS_SOURCE), eyepacs_subset_manifest_md5=MANIFEST_HASH)
METRICS_JSON = dict(environment=ENV, config={k: v for k, v in vars(CFG).items()}, final_system=FINAL_CONFIG, combiner_selection=combiner_json,
                    class_weights=CLASS_W.tolist(), best_single_by_dev_qwk=BEST_SINGLE,
                    eyepacs_subset=dict(**SUBSET_INFO, manifest_md5=MANIFEST_HASH, manifest_file=os.path.basename(MANIFEST_PATH), usable_images=int(len(df)),
                                        distribution_counts=DIST_CNT.reset_index().rename(columns={'index': 'set'}), distribution_percent=DIST_PCT.reset_index().rename(columns={'index': 'set'})),
                    runtime_seconds=TIMES,
                    split_sizes={s: dict(images=int(len(DF[s])), patients=int(DF[s].patient_id.nunique())) for s in DF},
                    development_in_sample=DEV_METRICS, held_out_test=METRICS['test'], external_aptos=METRICS['aptos'],
                    bootstrap={n: dict(unit=BOOT[n]['unit'], reps=BOOT[n]['reps'], ci=BOOT[n]['ci'], ensemble_minus_best_single=BOOT[n]['delta']) for n in BOOT},
                    eyepacs_vs_aptos=COMPARISON, claims=CLAIMS,
                    training=dict(total_wall_seconds=TOTAL_TRAIN_WALL_SECONDS, sum_of_model_training_seconds=TOTAL_TRAIN_SUM_SECONDS,
                                  per_model={k: {kk: vv for kk, vv in RESULTS[k].items() if kk != 'history'} for k in MODEL_KEYS}))
json.dump(to_jsonable(METRICS_JSON), open(os.path.join(PATHS.out, 'metrics.json'), 'w'), indent=1)
print('saved: predictions.csv (test + APTOS), development_predictions_in_sample.csv, training_histories.json, metrics.json')

def optional_hf_push(files):
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient(); tok, repo = us.get_secret('HF_TOKEN'), us.get_secret('HF_REPO_ID')
    except Exception:
        print('HF push skipped (no Kaggle secrets HF_TOKEN / HF_REPO_ID configured) - not required.'); return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=tok); api.create_repo(repo, exist_ok=True, private=True)
        for f in files:
            if os.path.exists(f): api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=repo)
        print('pushed checkpoints + metrics to private HF repo', repo)
    except Exception as e:
        print('HF push failed (non-fatal):', repr(e))
_to_push = [os.path.join(PATHS.out, n) for n in ('best_resnet50.pt', 'best_efficientnet_b0.pt', 'best_vit_tiny.pt', 'temperatures.json', 'combiner.json', 'metrics.json')]
if CFG.hf_push:
    optional_hf_push(_to_push)
else:
    print('HF push disabled (CFG.hf_push=False); nothing leaves this notebook except its own outputs.')

In [ ]:
def df_to_md(d, floatfmt='{:.4f}'):
    d = d.reset_index() if d.index.name else d
    head = '| ' + ' | '.join(map(str, d.columns)) + ' |\n|' + '---|' * len(d.columns) + '\n'
    body = ''.join('| ' + ' | '.join(floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in row) + ' |\n' for row in d.itertuples(index=False))
    return head + body

_te, _ap = METRICS['test']['ts'], METRICS['aptos']['ts']
TIMES['total_wall_s'] = time.time() - NOTEBOOK_T0
def _m(sec): return f'{sec / 60:.1f} min'
RUNTIME_MD = '\n'.join([f'- download: {_m(TIMES.get("download_s", 0))} (0 if the data was attached / already downloaded)',
                        f'- EyePACS preprocessing ({len(df_selected):,} selected images only): {_m(TIMES["preprocess_eyepacs_s"])}; APTOS preprocessing: {_m(TIMES["preprocess_aptos_s"])}',
                        *[f'- training {DISPLAY[k]} ({CFG.epochs} epochs): {_m(TIMES["train_" + k + "_s"])}' for k in MODEL_KEYS],
                        f'- validation inference: {_m(TIMES["val_inference_s"])}; temperature scaling + ensemble selection: {_m(TIMES["calibration_ensemble_s"])}',
                        f'- EyePACS test inference: {_m(TIMES["eyepacs_test_inference_s"])}; APTOS inference: {_m(TIMES["aptos_inference_s"])}',
                        f'- total wall-clock up to this point: {_m(TIMES["total_wall_s"])}'])
_ci_t = BOOT['test']['ci']; _ens_ci = _ci_t[_ci_t.system == 'Ensemble']
_ci_lines = '\n'.join(f'- {r.metric}: {r.point:.4f} (95% CI {r.ci_low:.4f}–{r.ci_high:.4f})' for r in _ens_ci.itertuples())
MODEL_CARD = f"""# Diabetic Retinopathy Grading Ensemble (EyePACS → APTOS 2019) — Model Card

## Objective
Test whether a small ensemble of partially fine-tuned, architecturally different pretrained vision models improves 5-grade diabetic-retinopathy grading (QWK), referable-DR detection and probability calibration over the individual models, within Kaggle GPU limits.

## Dataset
- **Data sources (this run):** EyePACS: {EYEPACS_SOURCE}; APTOS 2019: {APTOS_SOURCE}.
- **Primary (computationally reduced by design):** Kaggle *Diabetic Retinopathy Detection* (EyePACS). **The model was trained using a {EYE_SUBSET_SIZE:,}-image patient-safe subset of EyePACS rather than the full EyePACS training set, due to computational constraints.** The full labelled set has {len(df_full):,} images from {df_full.patient_id.nunique():,} patients; this run selected {len(df_selected):,} images ({df_selected.patient_id.nunique():,} patients), of which {len(df):,} were preprocessed successfully ({len(df_selected) - len(df)} excluded).
- **Subset selection (seed {SUBSET_SEED}):** done on label metadata before any preprocessing. Whole patients only (both eyes together). Per-grade image targets follow the full EyePACS distribution with a floor of {100 * CFG.subset_min_class_frac:.0f}% of the subset per grade (so rare grades keep enough examples, which raises their share above natural prevalence); patients are visited in seeded random order, rarest-grade patients first, taken while all their images fit the per-grade targets, then the remaining shortfall is filled with the smallest overshoot. Manifest: `{os.path.basename(MANIFEST_PATH)}` (md5 `{MANIFEST_HASH}`); the manifest, the seed and the unchanged label file reconstruct the exact subset. The image cache is keyed by this manifest hash.
- **Split:** by patient (grouped, approximately stratified on the image-level grade; target {int(round(100 * CFG.split_fracs[0]))}/{int(round(100 * CFG.split_fracs[1]))}/{int(round(100 * CFG.split_fracs[2]))}) into train {len(DF['train'])} ({100*len(DF['train'])/len(df):.1f}%) / validation {len(DF['val'])} ({100*len(DF['val'])/len(df):.1f}%) / test {len(DF['test'])} ({100*len(DF['test'])/len(df):.1f}%) usable images ({DF['train'].patient_id.nunique()} / {DF['val'].patient_id.nunique()} / {DF['test'].patient_id.nunique()} patients). Official test labels are not public, hence the internal split.
- **Class distribution (% of images):**
{df_to_md(DIST_PCT.reset_index().rename(columns={'index': 'set'}))}
- **External:** APTOS 2019 labelled images ({len(DF['aptos'])}); never used to fit weights, temperatures or the combiner.
- Preprocessing: crop to fundus → pad to square → 384×384 → Ben Graham local contrast normalisation → JPEG q95 cache.

## Models (ImageNet-pretrained backbones — ViT-Tiny: ImageNet-21k pre-training then ImageNet-1k fine-tuning — partially fine-tuned, new 5-class head, 384×384 input)
{df_to_md(pd.DataFrame([dict(Model=v['display'], Identifier=v['timm_id'], Best_val_QWK=RESULTS[k]['best_qwk'], Best_epoch=RESULTS[k]['best_epoch'], Temperature=TEMPS[k], Final_micro_batch=RESULTS[k]['final_micro_batch'], OOM_reductions=RESULTS[k]['oom_reductions']) for k, v in MODELS.items()]))}
Fine-tuning strategy: ResNet50 trains `layer2–4`+`fc`; EfficientNet-B0 trains `blocks[3:7]`+head; ViT-Tiny trains blocks 4–11+`norm`+`head`; everything else frozen (frozen BatchNorm kept in eval mode).

## Evaluation metrics
Primary: **quadratic weighted kappa (QWK)** on the 5 ordered grades. Per-class precision / recall / F1 and the 5×5 confusion matrix. **Referable DR** (grade ≥ 2 vs 0–1): sensitivity and specificity at a fixed 0.5 threshold on P(referable) = P(2)+P(3)+P(4), and ROC-AUC of that probability. **Calibration:** 15-bin ECE (top-label confidence), multiclass NLL and Brier score, plus reliability diagrams. Uncertainty: 95% percentile bootstrap CIs resampling **patients** ({CFG.boot_reps} resamples, seed {CFG.boot_seed}); they capture test-sampling variability only, not variability from re-training the models (single run, single seed).

## Training configuration
{CFG.epochs} epochs, batch {CFG.batch_size} (micro-batch + gradient accumulation if OOM), fp16 mixed precision, AdamW (backbone lr {CFG.lr_backbone}, head lr {CFG.lr_head}, wd {CFG.weight_decay}), {CFG.warmup_epochs}-epoch warm-up + cosine decay, grad-clip {CFG.grad_clip}, class-weighted cross-entropy (w ∝ 1/√freq from the training split), GPU augmentation (rotation 0–360° as specified, h/v flips, brightness/contrast ±20%, and a mild random crop implemented as an affine warp: area scale 0.8–1.0, aspect-ratio jitter ±10%, random translation inside the frame, bilinear resampling, grey fill — not torchvision's RandomResizedCrop). Best checkpoint per model = highest validation QWK. Seed {CFG.seed}.

## Computational budget (measured in this run)
{RUNTIME_MD}

## Calibration and ensemble
Per-model temperature scaling fitted on the validation split ({', '.join(f'{DISPLAY[k]} T={TEMPS[k]:.3f}' for k in MODEL_KEYS)}). Combiner chosen by {CFG.cv_folds}-fold patient-grouped CV **NLL** on the fixed validation-set model outputs, with temperatures re-fitted inside each fold (the selection objective is calibration quality; QWK is the primary *evaluation* metric but was not the selection criterion) on the validation split: mean-probability NLL {nll_mean:.4f} vs logistic-regression NLL {nll_lr:.4f} → **{SELECTED_COMBINER}**.

## EyePACS held-out test results (temperature-scaled models; ensemble = final system)
{df_to_md(FINAL_TABLE_TEST.reset_index())}
Ensemble with patient-bootstrap CIs ({CFG.boot_reps} resamples):
{_ci_lines}

## APTOS 2019 external results
{df_to_md(FINAL_TABLE_APTOS.reset_index())}

## Calibration results (EyePACS test)
{df_to_md(calibration_table('test'))}

## Ensemble vs best single model (paired bootstrap; best single chosen on development QWK = {DISPLAY[BEST_SINGLE]})
{chr(10).join('- ' + c for c in CLAIMS)}

## Known limitations
- **Reduced training set (computational constraint):** only {len(df):,} EyePACS images ({len(DF['train']):,} for training) were used instead of the full {len(df_full):,}; performance is expected to differ from (and is not comparable with) full-data EyePACS studies. The grade floor in the subset selection lifts rare grades above their natural prevalence, so the subset's class mix — and therefore calibration metrics such as ECE/NLL — differs from the real EyePACS prevalence.
- **Small held-out sets:** validation and test contain {len(DF['val']):,} and {len(DF['test']):,} images; the rarest grades have only a few dozen test images, so per-class estimates and confidence intervals are wide.
- **Distribution shift:** EyePACS and APTOS differ in cameras, populations and grading protocols; the change from EyePACS to APTOS performance is reported as measured and is expected to reflect shift as well as model quality.
- **Class imbalance:** grades 3–4 are rare; per-class estimates for them are noisy. Class weights (1/√freq) are a partial remedy only.
- **Potential label noise:** EyePACS grades are commonly reported in the literature to be noisy, especially for grades 1–2 (background knowledge, not measured in this notebook); label noise bounds attainable QWK.
- **Development-set reuse:** the validation split selected the checkpoints (QWK) and was used to fit the temperatures and the combiner. The combiner comparison uses patient-grouped CV with fold-specific temperature fitting, but it is **not fully nested**: the checkpoints were already selected on the whole validation split, so each CV fold's labels influenced checkpoint choice. The final temperatures/combiner are also fitted on the whole validation split, so validation numbers are optimistic; only the held-out test / APTOS numbers are unbiased.
- **Augmentation / micro-batching:** training uses full 0–360° rotation as specified, which may not be optimal for fundus images. If CUDA OOM recovery lowered the micro-batch (see the table above), BatchNorm batch statistics were computed per micro-batch (gradients and loss are otherwise exact).
- All images (including APTOS PNGs) pass through the same lossy JPEG-q95 cache before inference, as specified; this is applied identically to both datasets.
- ECE here is the 15-bin **top-label** ECE (confidence = max probability).
- APTOS has no patient identifiers, so its bootstrap resamples images, which can understate uncertainty if patients contribute multiple images.
- Referable-DR sensitivity/specificity use a fixed 0.5 threshold on P(grade ≥ 2); no operating-point tuning was done.
- Single run, single split, single seed; no hyper-parameter search.

## Privacy warning
Do **not** upload patient data or any identifiable retinal images to this notebook or to any hosted copy of these weights.

## Not-for-clinical-use warning
Outputs are experimental and may be wrong. They are not a diagnosis and not medical advice.
"""
open(os.path.join(PATHS.out, 'model_card.md'), 'w', encoding='utf-8').write(MODEL_CARD)
print(MODEL_CARD[:2500], '\n... [model_card.md written]')

In [ ]:
# ===== CELL 51 — FINAL EXPERIMENT SUMMARY =====
_e, _a = METRICS['test']['ts']['ensemble'], METRICS['aptos']['ts']['ensemble']
_best_test = max(MODEL_KEYS, key=lambda k: METRICS['test']['ts'][k]['qwk'])
_qci = BOOT['test']['ci']; _qci = _qci[(_qci.system == 'Ensemble') & (_qci.metric == 'QWK')].iloc[0]
print(f'Best individual model, chosen on DEVELOPMENT QWK : {DISPLAY[BEST_SINGLE]} (test QWK {METRICS["test"]["ts"][BEST_SINGLE]["qwk"]:.4f})')
print(f'  (post-hoc and descriptive only - not used for any selection: highest test QWK among single models = {DISPLAY[_best_test]}, {METRICS["test"]["ts"][_best_test]["qwk"]:.4f})')
print(f'Ensemble QWK (EyePACS test)             : {_e["qwk"]:.4f}  (95% CI {_qci.ci_low:.4f}–{_qci.ci_high:.4f}) | selected single model {METRICS["test"]["ts"][BEST_SINGLE]["qwk"]:.4f}')
print(f'Referable DR (EyePACS test)             : sens {_e["ref_sensitivity"]:.4f} | spec {_e["ref_specificity"]:.4f} | AUC {_e["ref_auc"]:.4f}')
print(f'Calibration (EyePACS test, ensemble)    : ECE {_e["ece"]:.4f} | NLL {_e["nll"]:.4f} | Brier {_e["brier"]:.4f}')
print(f'APTOS 2019 (external, ensemble)         : QWK {_a["qwk"]:.4f} | sens {_a["ref_sensitivity"]:.4f} | spec {_a["ref_specificity"]:.4f} | AUC {_a["ref_auc"]:.4f} | ECE {_a["ece"]:.4f}')
for c in CLAIMS: print('  *', c)
print(f'Total training time                     : {TOTAL_TRAIN_WALL_SECONDS/3600:.2f} h wall-clock ({TOTAL_TRAIN_SUM_SECONDS/3600:.2f} h = sum of per-model training durations)')
print(f'Total notebook runtime                  : {(time.time()-NOTEBOOK_T0)/3600:.2f} h')
TIMES['total_wall_s'] = time.time() - NOTEBOOK_T0
print('\nMEASURED RUNTIME (this run)')
print(f'  EyePACS: full {len(df_full):,} images -> selected {len(df_selected):,} -> usable {len(df):,} (train {len(DF["train"]):,} | val {len(DF["val"]):,} | test {len(DF["test"]):,}) | APTOS {len(DF["aptos"]):,}')
for _k, _lbl in (('download_s', 'download'), ('preprocess_eyepacs_s', 'EyePACS preprocessing (selected images only)'), ('preprocess_aptos_s', 'APTOS preprocessing'),
                 *[(f'train_{k}_s', f'training {DISPLAY[k]}') for k in MODEL_KEYS], ('train_wall_s', 'training wall-clock (all models)'), ('val_inference_s', 'validation inference'),
                 ('calibration_ensemble_s', 'temperature scaling + ensemble selection'), ('eyepacs_test_inference_s', 'EyePACS test inference'), ('aptos_inference_s', 'APTOS inference'),
                 ('total_wall_s', 'TOTAL wall-clock')):
    print(f'  {_lbl:46s} {TIMES.get(_k, 0.0) / 60:8.1f} min')
print('\nArtifacts in', PATHS.out)
_art = ['best_resnet50.pt', 'best_efficientnet_b0.pt', 'best_vit_tiny.pt', 'temperatures.json', 'combiner.json', 'training_histories.json', 'metrics.json', 'predictions.csv', 'development_predictions_in_sample.csv', os.path.basename(MANIFEST_PATH), os.path.basename(MANIFEST_PATH).replace('.csv', '_meta.json'),
        'confusion_matrix.png', 'calibration.png', 'training_curves.png', 'class_distribution.png', 'per_class_performance.png', 'eyepacs_vs_aptos.png', 'model_card.md']
for a in _art:
    p = os.path.join(PATHS.out, a)
    assert os.path.exists(p), f'missing artifact {a}'
    print(f'  {a:28s} {fmt_bytes(os.path.getsize(p))}')
print('  preds/ (per-model logits, probabilities, labels, patient & image ids):', sorted(os.listdir(PATHS.preds)))
if PATHS.big_in_working and CFG.delete_downloads_at_end:
    for _d in (PATHS.data_root, PATHS.cache_root): shutil.rmtree(_d, ignore_errors=True)
    print('removed downloaded datasets and the image cache from /kaggle/working (CFG.delete_downloads_at_end=True); outputs above are kept')